# bell-transport-bench: moving a Bell pair across three processor graphs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rickory2021/bell-transport-bench/blob/submission/notebooks/bp.output.submission.ipynb)

**Our submission to the quanTA challenge, UCF IBM Qiskit Fall Fest 2026.**

**What this notebook is.** The bell-transport-bench repository, compressed into one self-contained file so the benchmark runs anywhere, Colab included, without a clone. The repository is where the work is done: a teammate drives it with `make` commands, shown under "How the repository works" below. This notebook packs the repository's package, its recorded runs and its config files, writes each config file out where it explains it, remakes every run it reports, and sets each beside the run the repository recorded. Every config file, circuit and result below is the repository's own, carried here.

**Overview.** Under one simulated noise model, our protection raised the fidelity F of a Bell pair between two distant qubits on all three processor graphs, keeping at least 97.7% of shots and using fewer two-qubit gates than the organisers' baseline. [Section 4.1](#4.1-The-nine-cases) gives each graph's numbers.

- **The task:** create the Bell state |β₀₀⟩ = (|00⟩ + |11⟩)/√2, which the challenge writes |Φ+⟩, between two fixed qubits, the ports, on three processor graphs, and measure what error detection with postselection buys and costs.
- **Our protection:** GHZ disentangling moves the pair by preparing a GHZ state along the path between the ports and disentangling it with gates, which leaves the path's interior qubits in |0⟩. The vacancy read then reads those qubits at the end of the circuit and discards each shot where any reads 1. It detects errors and corrects none.
- **The noise:** one model from IBM's Boston calibration medians, the same on every graph, simulated with 100,000 shots per basis. Every result holds for this one model.

**Where to look.** In Colab, the Table of contents pane in the left sidebar lists every heading; in Jupyter and VS Code, the links below jump there.

| Question | Where |
|---|---|
| How did each of the nine cases do: F, acceptance and cost? | [4.1](#4.1-The-nine-cases) |
| What do the transport and the read each buy, and what do they cost? | [4.2](#4.2-What-protection-buys,-and-what-it-costs) |
| Why do the three graphs differ? | [4.3](#4.3-What-the-graphs-explain) |
| Does every run reproduce? | [4.4](#4.4-The-notebook's-runs-beside-the-recorded-ones) |
| What does a run do in plain Qiskit and Aer? | [3.4](#3.4-How-a-run-works,-in-Qiskit) |
| What do the circuits look like, written and compiled? | [3.5](#3.5-The-circuits,-written-and-compiled) |
| Why not leave routing to the transpiler? | [3.7](#3.7-Why-we-wrote-the-transports,-rather-than-leave-routing-to-the-transpiler) |
| What noise did we model, and what did we leave out? | [3.2](#3.2-The-noise-model) |
| How does the protected case compare with its alternatives? | [Appendix A](#Appendix-A.-Each-protected-case-beside-its-alternatives) |
| When does readout favour the textbook GHZ transport? | [A.4](#A.4-Readout-and-the-two-GHZ-transports) |
| What do the results mean, and where do they stop? | [5](#5.-Interpretation), [6](#6.-Limitations) |
| What do we recommend for future systems? | [7](#7.-Recommendation) |

**How the repository works.** Four `make` commands drive the benchmark, drawn in the diagram below. Every input is a YAML file checked against a schema, so every result traces to a config file, a seed and pinned package versions, all in the [repository](https://github.com/Rickory2021/bell-transport-bench/tree/submission).

| Command | What it does |
|---|---|
| `make workbench` | Gives a teammate their own workbench notebook: pick a graph and a noise profile, build a circuit, see it written and compiled, and save it as a circuit file. This notebook's circuit views come from it. |
| `make benchmark` | Makes every run a benchmark file names; each run records its commit, seeds and package versions. |
| `make run` | Remakes a recorded run and reports any difference. |
| `make submission` | Renders this notebook from the recorded runs. |

<img src="https://raw.githubusercontent.com/Rickory2021/bell-transport-bench/submission/docs/assets/design-flow.png" width="480" alt="The workflow: processor files, noise profiles, circuit files and benchmark files in config/ feed the workbench notebook and make benchmark; both build the processor and its noise with NoiseControl, which feeds CircuitRun to compile, simulate and score each run; runs land in workspace/ or results/runs/, a benchmark's runs in one folder with a comparison table, and the submission notebook reads the deliverable benchmark.">

**To run it:** Run all, in [Colab](https://colab.research.google.com/github/Rickory2021/bell-transport-bench/blob/submission/notebooks/bp.output.submission.ipynb) or in Jupyter on Python 3.12 or later. Section 1 installs the pinned packages, and the runs took about four minutes on our machine.

---

## 1. Setup

**Run these cells first:** they install the pinned packages and unpack the code and recorded runs this notebook carries.

<details><summary><b>What these cells do</b></summary>

The notebook needs Python 3.12 or later, as the repository does. The first cell checks the version, then installs the packages the repository's `pyproject.toml` pins, into the Python this notebook runs on; pip leaves any already installed at the right version. In Colab, if it upgrades a package Colab has already loaded, restart the runtime when asked and run the cells again. In the repository's own environment, which uv made, everything is installed already and the cell only reports that pip is absent. The next four cells are payloads, each one line, which opens folded under its title in Colab, from the repository [`Rickory2021/bell-transport-bench`](https://github.com/Rickory2021/bell-transport-bench/tree/submission) at tag `submission`:

- **The code:** the `bellbench` package, `src/bellbench/`, exactly as the repository holds it (19 files).
- **The nine cases' runs:** every file of the snapshot behind the nine cases, `deliverable-2026-11-17/20261117T152026Z`, from `results/runs/benchmark/` (27 files).
- **The extended benchmarks' runs:** every file of the three snapshots appendix A reports (113 files).
- **The schemas and the core benchmark file:** what a valid config file and run record must hold, and the workbench's defaults (7 files), shown folded after the setup.

The sixth cell unpacks the payloads into a temporary folder, and lists every file with its SHA-256, so each one can be checked against the same file at that tag. Nothing is written outside that folder.

Every config file but the schemas is written out where this notebook explains it, not packed: each `%%repo_file` cell writes one file into the same folder, and the code reads it from there. The schemas are packed instead, since every config file and run record is checked against them when it loads, so they must be in place before the first one; each is shown in a dropdown after the setup, with the core benchmark file, which only the workbench reads.

</details>

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(f"This notebook needs Python 3.12 or later, as the repository's pyproject.toml requires; this "
                       f"kernel runs Python {sys.version.split()[0]}. Choose a 3.12 kernel and run the notebook again.")
%pip install -q "qiskit==2.6.2" "qiskit-aer==0.18.2" "qiskit-ibm-runtime==0.51.0" "qiskit-qasm3-import==0.6.0" "openqasm3[parser]==1.0.1" "networkx==3.7" numpy pandas matplotlib pyyaml "jsonschema>=4.20" "pylatexenc==2.10"
import jsonschema, matplotlib, networkx, openqasm3, pandas, pylatexenc, qiskit, qiskit_aer, yaml  # a failed install stops here

In [ ]:
#@title The code payload: src/bellbench/, 19 files
CODE = "..."

In [ ]:
#@title The nine cases' runs: 27 files
RESULTS = "..."

In [ ]:
#@title The extended benchmarks' runs: 113 files
STUDIES = "..."

In [ ]:
#@title The schemas and the core benchmark file: 7 files
SCHEMAS = "UEsDBBQAAAAIAAAAIQCRl6z1+QEAAGQEAAAsAAAAY29uZmlnL2JlbmNobWFyay9icC5kYXRhLmJlbmNobWFyay5jb3JlLnlhbWyFUk2L2zAQvftXDPSwLdiO7bbZ1oVCCdv2sPS09zKWJ7aoLBmNvEso+9870obg0JDNSZl58z5m/AaUs3s9bDqyapzQ/9l0c9ljwPJUKZXzVB5wMllmcaIWYiEz2JFpYSdvOGEz6wJxC1+LDOBhJFAjGkN2oBsGq60UkInhbe8Ub3Y/v93f3/36cVdOfZ7gL5BImXDvWghSDaMnEsLZO0XMzsPkejIMi+3Jg5MZ6zRTBOy1oRw6x8HZoqmabQ7sEoty04xes7PAo3tiIZTwllTQjzocAI0QlXCHagS/WNCcxpwf0Aq55xvoxJSJFoNHy7PzQcRBB4bB4zyWQrlL+d7nsAW0PXzOE4kYCyJEfWTmHNBHyyHpjOTpS0JJJv1IHjuzWmkupKtGylTUdVHf5iBhTXI5wZMOYyJh8RZoOLz8WbpJM2tx6Sn65eixqfKqquIWxPksG5RcknYQDYbvgMIh3tH3QN7Ltt0esHNLgKqsqrrM0mArNPLL4k5+M1EfC822ruttFkO2IlTAX3m2oLupqWo5hdJeLTq0p00Wp1a6YLu+3POKQS4VaJIE6A+XaM7717nGg2TunNHqEtO6e4knfZjHcPHdQp3DeUrW02IwyNal2BOa5zW8uQaPiocz+Icj/DzhdYmPr878r3N7nFnnv67y6ZWJo8Y/UEsDBBQAAAAIAAAAIQBGP9BbegUAADAPAAAfAAAAY29uZmlnL2JwLnNjaGVtYS5iZW5jaG1hcmsueWFtbKVXbU8jNxD+nl9hARLHaZNASjkR2pOuV+5F4milq9oPiCJnd5L14bUX2wukFf+9M+PdzSYhHKj7JVl7PDN+5pmX3RapNVM1G07KgU9zKORgAibNC+muB3NZ6N52b1v8KbXKZADfSjcydA53uqde87mxsAZEu5oIKYwsIBMeAmq0U+Eq48WdCjlLprYopVPeGhHkRIOQqbPei4A+iVeZTf3w19Ovnz+eD4osQSVpUCh6tDcQpzLNUSOqE8qjmVS5tFJBTBVqqUwGjmxb5UGUztLqiZhaR5rRqltyMpUeL1nIEhXyfi61BjODXS+MIjdZwJpgUSmZlCbDf14VlZbk0kC8zyG9JsdlEL6UBh0BnXnU+IpOx3vDvfIhESpDjx25qW4qSATgXVq1C6V0F+0jTqgiyqFC1udVQBxzQDWrLs+cykRZ4b4Ke2xIGeFdivHTmq89RGODcp6gAtXAWIOOKg3cMnotQhFUbWXmB73eTmTMWGzlIZR+PBx+w/j1ax5ZNxtmTk7DcLQ/2u8fjIZxY6sXVNAwFr8sae1l4FOnSrrvWPyR10B1jRfyGnxC3iknfG7xXoTThFEAQF7QK9+EQYyoxXeCCl0O8xIt28k3JFDPwU2lHGRjcUHcTISWCEyCZAlkhy0krP4qqiePLnsyyxR5KfXvzpbgggI/FlOpPSAmGUz9uCdEgPtAv/iPbfrglJnxQqHMGUYo5GNxgAteV7MNkqUMARzisfv3hez/s98/vqx/+5evd3Z7vXLhAR6gW0RNS2B+RQMMm7ipMJORj46IQAscTzp2wuh2wD5/9+X0559o622MhAoDVr3jYIoh3x7yVYfk/RZuMHZj8e/KNsGw9UC+EaiPOPfBATBYwss53hvjhYlDvi18kcbfgYvRlmYecpLD/ANJ2Y1xqnQmro29e9RDdoFwpng+Bg8zCWGkUGMJaQoIQsT52MlDLFuUEnNCZNAJmTIBZuCa6OKBIsa2Jc8jdonjLXcXelHIqVsgwD1veTF1tviuuf34Ku/j6+Ho+PD46M3o+EdcpwBu8IC26sLznNqZxJKNURA+lfRu0XGsMxDDgw2iyQ4qRcaA9l3XpXNy3jj+OUDhI1ACr0sv/LcRrvM0LnWyFX1OGl+T6N9lLfVUckaJ5ZSpdVdm8bIC0tt+Z0c0mFFDwNxgNDiXkDidRhILejfLxhzPqdXZIvvaA6jNga90iOVtySBRIrW6Kph+sWst98pY90J0jKJIpCLvauTjsyFv41ODuZ6/LPbQyjHW35ViZqBUjKKpigk45Nh9qiuP1P7SUnZxpGVRNw6rlInPElE20mWNNDfVRBG+3I4TcSt11bLmudzZzKD4RBurq4I692/T9WUh+g1IdUonnXx+eEqeQUm6SZRQ7tcvo6RG6Sn1q/oZmCcqePdh9FYivJDpFIH1cK5F6XmwbwId8wbCFZLHutafi4ZyW6bSeuty2ftCZVc1368KkL5yUIB5mQ4fJPpiZldIahVkrBRPnOS58ZEK3BaXD3Z50OLxMQ4wTwyicTB+/+nd2dnp+cdTno2pQEURGm5Zbi/pTEVYK2qbawMmdwHk6smSK1TibBVqkzxj+1Blc9Rpm3mWzmNxMPB4rX9BbY8DLlf4hX//r7qTypeUd5InnDmMTa1enasx4wR+BfzAm6P9g6O+Mr6kS4jCZjRGHtL+UV2xMf+K0jrp5stcjKJvSPSYRXNEx02sVumawtikIyGwVGKHpfFBY6zjdEfFed6ucVPgpSWL3L4RpRA/oQZr+dkdLpqE6Uw07WIzaBw/q4vWbRPH6Tk3Qm6eTeOk/kgtEJn07I61IMdGm38x57lpYjLFph3ubOeo76YYfyEsVIGh210w0HHImF/2/gNQSwMEFAAAAAgAAAAhAGe9q6fQAwAArggAAB0AAABjb25maWcvYnAuc2NoZW1hLmNpcmN1aXQueWFtbH1V227kNgx991cQSYAkC48nm6IPdYACwfYKtN0WKfqySAuNTY+1kSVHlDvxFv33kpI9l3R28uIJJR6Sh4fUOVTONnq9XPUFVS12qqi0rwYdilF1JjvPzuEPZXStAtJ8d7ohPmzfebyJPiU4i7DxOgS0MJ2xTcyM1ntXIZHz0LkazSVB346kK2XgeVjpQDkoAgV+sOBR1QTsfVW7ipbffPvw4/e/FF2dA2EVtLPEgLegbA1fXhfwe4uwjokqj/C+R/vb/cPP8AVfd4LIaOjZlS/gi6qCGWHTqiChqIB7+TKeqjkmqqqFlSJNnKB3QUm0GChwkN75AB0qGjx2aAOnSGiaHLQNEqkyimJJDOdxrSlwXKs6rKPrHV95zY+cSqkEbmO3TgSrkcMbLBjpXYvVE3ECnDL1ykKj0XCuV2Fbd688YR5znIFbRSnnmfdLIW2mWk52wYS2lOYU9jqatAXy1XKFxqzQVu0sACr6MWewTauZLWFR8Dr+Hy3+zSWrbRaNNgjGcTuLLLtISivhrA2hp3K5/EjOLib9Ob9e1l41YXl7c3uzeHu7TAdnWdDBYAnv9jCzGqnyupf2lPD+s8J7LbscNjq0sXxllWEBTsRO96ZeUOuCZBzGngO71UeWXebxedAe6xI+CFk5GLUSxG2IfIuZg3Xcl3zO5jFjdWlJVplfvevRB41UQqMMIRNTY0NlBhDwJciXf8XIFLy262jotP0J7Tq0JbxlA5lh/ZmbvWIiPNNy+ecHtfh0s/jqcfouHt9cXGZZv8uAHaSWhHTA6QMHSDp5HngRsOa8CEIMsanidpeaL+MRBdIpz0LtdfV0IEUWFW+JGOLCY8P9P1/GkpdSxRkfRCaPJHEvneCZa4eOdS8hoeFW7oEfhRUaBXbbmSPQsjQioGsO5ySVt2kd4esVBbuJ4zXC+jpZ1CyGI8G/XkQbwA9uc7DzJEAUXxl/zgkqu5Pr1IQ0vX78/5TON2VK04xOwaq0SKLAMQTWC8WZ3TU1TSovK5LFxqKHoJ64WsujdCdbRodxMiWwHBrPOf71KYog/X4p9nQ5zY4YXs3PYzSemoso5gOtyl/UK/xzjPR/pytzdXxtPws5jnN5pCPfecQ4fkBqZF9ouTP7It7wPl0N2oS0d4NXlmSppxnhmWun/Rt5uY58KDuGVsB2j1DrBlPDk3Wbk8qdop6SzvbVSw+n3Xv4hLU192Jegtu36Hm3/vbehaRufuUHy2/U9hHbPRB7zG+fCLiKlbIAjFrvZCDzk1RwzXkZiS4mZ4zbJGfOdLDMBVXOY6IpIp/g4z9QSwMEFAAAAAgAAAAhAGWUBfpNCQAAXB4AABsAAABjb25maWcvYnAuc2NoZW1hLm5vaXNlLnlhbWzVWWuPG7cV/a5fQcgGbKej1yL5UC3awm3Swmi8TZtFkMTdCtQMJTEaDWWSY1k19r/n3EvOS4/V2g4K1B/WO3zd17n3HnKfiNQUC70czbdDl67URg4Lo50a7uUm7z3pPRE/yFxn0itXreR5Wo/RavUXvH4qTKEEj4itNQudq0Tc0OdfTOGtyZ85HOiU97pYOqEL8U/t1to/c8Iru3HieWZSN/r6m+9f/e1muMkSrE29NoX48sVQvBSukFu3Mr46XKxMntGRJPadzEuIVVa8LWXhtd8nwq+U2KhMy0KYd5iRYquLQmXi1Z9fi4Vcq8FcpmtVZPXRrOHbcq69ExLjmbZQATtSU25zZV3CwzRtdoUopNfvlFjCP2QXFDaY39cKbqW2dOJO+1U1kSrnjBVLK7crsipTW5NLqx2c0rUM+tIgPtOVhN65kAtPZuDEQsIwFgyhZYrTHZtr7FIW8Lh1z0RqjXMDRDVdB7VhvyxzXUuR4jv6xnHKWugU1IQ041QRAznXODnK7UhdYANJpAUqG4TVzpfZfogDb/UGkJFWUZgRRlNk8B2LCcPQYS7nOtdeKzcUf1f7MJ4pl1q9Jb9eB8eS0DBXwSUhv6QKYlJt05KikZrNlkzyBkpBfYh13pYMHzdkJH8j4aYAk4VWeQbY1cKAMQ0ZuTNsE07bqMILOQduMCJ92ELGKCBpTxFgg8mP14Cv825EP2chS4bbPQCmcyd2K7iSzvQ7AzgtFsoyTABLT+MbsVKWTKEApQb7eNDT6PD0wawmkOyCLk1eBOPgAniEMi2OD3u9pyG5p6K/8n7rpqPRL84Ug5jyAM0oswjy6Gp8NR5MrkZhot9DHuVqGpK4gk2v5bWpuF1VKc/SXSfhxbzUAci30i6VjygM6zcmA6QX1mygoN9vIcfMf0G69ax6WyLxsql4Q8FPRC7nKk/EWhcoClbBhwrxT3AQXHPXkxmcCW1k/p01KAEEqSn8nzsF0zO1cNOegC/fe/ofv7Ew4APZxQMbXXyriqVfTcUEAxGv7cVFuZkrWy3Wm3IzFWN8ljY/c+ZWeiQNPPTsP+zzP8HpzzDTAH//GAH4lO/DJ6lGGdHeFj1GAy2vhQoW823WkpiIrLSSfHXHex7yHBvRjPK3iMWx+hKiA4YJVwVJBXnAC2OtuIrjyIHW+LA+RBVk35sJVt7FwSPVp+LDU4QeEH4y4piOWpP9+7itMu94dQxqvZKAN6tSZNb49cgoQnjTH7juc12ipI6Y3wGQwqOhFHxsErIeZoaEdzyKDagxtBWnEK4b+w9UJaT2CbEootnlaH9uSB/js153G6kftndc9X1eLkPbRRPONWqmpZJJA9xwaNu1sGURuuscWbzaSLt2KGfpmtdVvWm+h/+GF1PrjRz8dzz4/V38f3D3xVNKMi4Yx/awYwkAVElO6P/HQXRJFe8k8geEfQH8nuQM150WnjAb6TT1g+7d6qLXURy35aTqx5e6MZqVol4WMokVa44MLosJ1ZjRVZHlEVzqWnrCGYx7U9qU8Kq59zE/q6hfvdcx+/vkWHERPyH/r1YpLtpkHfnR1wqRLjtjs0jHwKyQW1ghYZDM4GQ4pQYdHFCida8Lsws6nsm2yllnXBGpY91s5REfhZtiPQB/UFUloCjmugDGQ2NmpYB/H+AeTvVyGVkEW0BTLtIUN7xYANJcOkdRTddyiXKbIvfmyGmFbtnkbRIZ/GdWCxb2QKl8yyRtoOebARKdShinjajShvdfLH0BOmzPA7LiCgEa5IjDMVNjZuRbRftRwhqfPa4NVOtJbNZpZkcpcJAG/84+fHk/wM+r+JOz4LSzD3T4FjgKFLejS7MvIP45vmKfpfSgL+5YAZ0vDrAXnUi3MLk86ysQndpVbRh9nKadrVFZJty5ofqPJvBZ2hHaZs6r7Ykc7tOVRNBk60oyZVErabMdLhikYK3IUNyUed7w9zrbcSOk0oO6Vahh8Al++8ei8sTgmNFxSX6f5qXDteZ1l941G/oFBNKBHLrLfd9PZlbl8j07c0a2o/dezVDqV5LqfByigmJKP+Ou8qlZfyzrTOD7t5OOf1dmJ3LDtTkgMlxtJ2j+4IRK8Klc3Y0YXxNnAklyXsP1k9gxuYb4UElbnA8e3T4f+NHt5MWwfw4YFYOpzDh0z1krrh62Yq3UNlQZOg3VTfmdAlLGnHGTa1HQ3Qx3HMv3xwIU+HbyaDU7ITuj4ku3xyUVVSatlsfrNbFPDXpg1TaX6JUvlQWwm9UAqzusVi2AHYCMznazyYzyYDYOaHKzcfie3NVbLkHqXH0ToiuhPXMYlH+17ZxWdLHGA5o+nUVgmjR5G+K1A28at/x/Igbt68SRepXBv41641PqTT5KPSrop1hT/6Z5kwIUQKPDq0mg7I1IFzikTD0TyucTAcp59SJhihIIaFt5elgrnWqev6pLQytNov5HiAJ5a2Ni8kAJOiTsZAUT9o2SrrSnaGr/dZiiB5vpWdXEq8qs8JREfuimTdT+kCLQFazPTNmpU9QQUccEV6/pBcecO3puoMBOnbqQ/BD4ZKf/IKnXiGxz3URMDL0XVeSYQkS3KnRPwKo6vU0jpbVyz9/aq01T5I9rQasSMBtNAolIoqyEHkCo4OyrSnC5DpyqAnz2A/e18I9FY1W7vzazQaOLh9QvNhdYC5OT+o7PlN5FTn/E3ev7vnbtewEdMeydzegWpRLkwgdUb1/fLhMDbpuz8PTSefzhovCpFKA59QGGzDyTK8vJK/DC5LnZPZL+Hz6UHefe120ZYNzxDY+ZAr+2dKz3UT90w+qJH5UgVwtPuYrbPejXXlikD3LKeSL1Zxv2iWbRqsjnXYNacR2fbfk7JC4zhMa/BCOdKZkftukmb2NdfcXpG0tqk8/ngcQ3/89AED1A/G9x1H4SifBJ+AmEnRjaJ110QiP7rZB1e9C8+WbZenWpsFQ3UCrH7uAF5/mPifiJ+urP1FcLZAQuwGhTOZMB/iMSh5+P/CisURjOqX7TvBQpL8UfxHb2sxjhUjj7UfwOHz9BmfCHDfDt8fArQltZhD/jJNQxOpkb3p20a6n3kdeb/4O86Mk8Dxe4gdCY/9BKBXoqvGu/o2Canw/FBzR3B85R1fv7+9BloHrRPaN5h6tvqEmIelIRtUhw7u4/UYl20M4r0l71yaK4hpyXER4X73u/AlBLAwQUAAAACAAAACEALxLi8lQEAACPCgAAHwAAAGNvbmZpZy9icC5zY2hlbWEucHJvY2Vzc29yLnlhbWydVttu4zYQfddXDJIA2V3IVpK3dZ56QYsCBdqil5dFNqClscWNRCoktY626L/3cCjJTuJki/rBlobDuZw5M+NTKq3Z6G2x7pa+rLlVy87Zkr23bjmotslOs1P6SzW6UoH9pD3rxHs4Obz1Tu6tyBqmWUqtrbjJSQcPe/f9Gg/yRmFnqbMOT8pUIilt3zXabGnrVFcv6Y+aybqtMtqz8+cUasecDnHJMQwGqJTWMXkO1zA0kIXIJSXSUKNN70S00Q3MbeSKVy2+atXxkr6rubxDOLUKMOg7ZaDKTeXpjcaXNoihYpcfBMvVFpAY1cZgHatmToxVWcsxUCg5h8HOeh20NZ42QIM/sxuSNsySAVY5YgS6hsvAVQr8bcyOyhjY7R5JbWJ8rizW3DRrNmVdpKr4ZTfktKs1fLsennY1m+gpZh+zbqyq/DWhjsEX8fs2XcQ9mCz3AAg4jjfsYJ5JI9UHjUvLLDtLLFnRSR1C51dF8clbsxi5gzIVlVObUFxdXF0sLq+KdHCSBR0aXtGvjwmRVexLp7uIzIq+2fPl3L9CklRtkAR0oDWHHbMRtXQFUYahgzO7/gQ0M8f3vXZcregDagWgG7WOXJwTzFEAwJHPLsVVnup7k6mqktKpBtF37IJmv6KNajwDj4o3fpURUH0I8RdP4twHB1qIAPz4mc021Cu6hEC83OoqaT9BQA6Rva5y+k37u/RiKn4Q4kh1unrwupzotjxwqk3gLbvJq277dkUXWdbtA8dZROGI89+bfpsLuPc9Gh7sj2QTgdAnXrtOzIpVEOq1yoEznS7vRG/P0vVAj0M7wKNTIbCDx/OPH9Tiy8Xi/c34u7h5d3YOHanQkRDjKBDinEvfMSmf2FCrpgHEGAv6M3pycn2GGoOqp4WUqYg1OsHBXPkXXHjbu5KlChGBypZ+HnTzXS9z7n+nKJQ74v+HONxioNNgCXNAMZYdphCoGQuAMRfqqKHi8MFsmsaaVMvXtm8qujN29yoWifNHAvkF85tNwJwCdZJaHiMAJjIJr0eqxBmGbjQhztkoajAqgF0jlD/ERzmnhombPwVu0UZX8q7lRR4n5bF5k+ighXV1Mwpf68uk8Zj26TP1Xfo8yvhovyGpx/2W0wUFS4YWdDlO+jjklwdWn0A9NfzJrDJtgxdD+d6pXSxtaYG1NrJ73zzkNLzNqfdYENY0g0SI+d07kJF+BPMj/u2xDbMZt8w17XQAM2TEt/l0O9oEn0j1wbYq6BIdOEDrMKmnNUyfp5UcperhiDRVmf5Olkzfrtn9k1Eaty+0Yty9I63jEkhsjZvNSbL0LZYguk1jofQ+gLAV9Z38Z4Bov3TVqD4SqQWxE3+j/elwbnsZP/6/UfdpqlOSL3EgJiyL5UjCf5oKwckfgGf7LZVSxwWuIhSLJNiCGohhIFWi0CbxUfBAq7KWvzzJKDx8LaNLee+Nvu95FAXX80stesiF5zx4zoGvQ/MvUEsDBBQAAAAIAAAAIQB+qq3wzwcAAMkYAAAcAAAAY29uZmlnL2JwLnNjaGVtYS5yZXN1bHQueWFtbMVZbW/bOBL+7l9BpAV6Cyh24217iAMsUOztHQrstrvt4falyBm0NLa4kUiVpJIa3fz3e4aUZMmxo2ywwPlLJIoczjycl2eYJyI1eq02s1U1dWlOpZxacnXhp1tZFpMnkyfiP7JQmfTkBObgYTBDKC1IprmwtRZrU2RkExG/uxnG3Gw6nQpjIefG2CtXyZS68YXwOQlH3iu9cULqDEtTYzMnzFoYTSw1ETe5wgbfKpvWyr+vNcuyihXiFRJreHNLq1oVWLq2phR/y0zqZv/47sObf72dllmCXVKvjBYvv5qKf2NXXZcrshBhCfJgBasSDXjmhFNlXUhe4Kapuw4bpabWPrwmQtM1WZGTpWlPMQEA0ysHebIxRMiNVNp5SFeQGgCGPaSF8q0RbCfLb4ctySyOrmR6NZ1MnsZ1C3GSe1+5xWz2uzP6tDkuYzezzMq1n82fz5+fns1n8cPJxCtf0EK8D+CwNpOMXGpVxXYtxLuIL6xdq4Jc0h0EP1GGP2m0jJ8MbGAlK2uuSUudEhTz2wryzep3YDux9KlWlrKF+BjPf7k2tpS+EwPUjHK0hMN5awrskhuWvZIY5B27fTMlN1aWjj1po5zHOSUBLU+adSkrKJxFrZKAalZjYJnVNpxZ0j+/pKfz5URmmeJhWfxoTUXWK3ILsZaFIwCd0dotJkJ4+uz5L56Cic5bABMGSqW/J73x+UKcYYChizMH2L5uwip8F6stztXBmMqcWmK9rklU0ucBUna8XLqcfZ6nVdI6YnfTMNdPe2o0SPNAD20WlAQJl+HTfSby92o3Gt5FUGUhvjy1tIaTPZkFHGYMwsltM4XFY0ofjoTX4Wxg7rP/ulzOX75afHx+ei5P15dfXr24ffqMF/OR9qFUsGpDtsUS51QuxPP4Kj/H1xfz8xfnr/4+P3+JcXYPdwBidl/gY7eIWhtzEABkMH1uieK6cfR++SURv/6aiN9+u+yjs30rS4btC2nWqD8tWNX512J0h8apS/K5ydxjzyhIaV/2oOB81mgEKHimeI30VLMbdeAAD6SgJhannSBZFO9w6h/3Tz/Ad3J7uZt4UOX9Zbx55zSNyeNac2KOk/9vig+8ncrKb5c9zzuofPjOpSMWKD+054oqj4wXshwSkgmx/U8BU2qNHZWm7KJNasj2BXLEGvlgu7MwOpW0Vm67MYTMG0+li9kn/mqtPtXUDHtbU/dFxbGDTjwZ+tggZx8It29OG6nh2HJZUQw35ZriciFWpNO8lKjyYoOMgoMMZnG1b0Mzra1F0Iq4zVScLboiw0C+q0j/9PrDD2LOmbHZUPYqLx6LutQCine1QaCY0IWYHxH19V7x5npXF4UwtUclIZzTdop83VTIuKXhk4D32UJBflQWWU9unbhmNiSkE7Ac0ba9iPQjcpFrRTeMxxoeLFpnbcA/S8ScnbJR8q4HcqkIHjgokweO4i1//zZ+hjX0uTIWLublFemORTAZ8gHz0mRUPKCMWJOSc8Y2dZoLqyzwB8XTWpVRoAG6jaVH15p2m3sA4F/QYWROULCrSpHRJcAjLWqHEvtDV17aBZ0pi3uDLIZN93oAt/jrofepXgWitFZUgJnAS2q67E0dA6oDZw+u+IvSO0uDtkk/FyRcPJuXedLF/aDkJr16e3s7kB+0HsmJ8RcM24O8ndHzjn1896AbckXySxyLCQpmyyY+liVJV1sqkS7i57GMPgTzMJS93TojPraOc6KRF04u+wbfq9CDJCBrWGbVS7il8j3KcLSsQPOVXKlC+S3iF4knVMHgAlEack2oKpzm/jj75qLJXaiZId+GDAAzEyFXjpMtFNfQACGlaTrYu3+KzGqYkB9IOB94POTcQS3uer9d1ZuOMb2zhtEFwn8PfejRnbukgb+F1MZ9Hxo2xHoW2z5O/iw+kiBOyYVcUYHA5H6KLUDFqMaTobdSO+4yEEsZySLmxO1jk14nboQwhb1G5gRF7pnTNk8HwHvf+oncwXeD5w0A5KIxFW8CZSmMxsGhmzWGGY7sNfeoN5VKPeLAtY05OujIfcCYI2tDlg1l/94uvF8bC3hoEsryZaAiTdM3RkPaTrAr/M9EWkjnFOrCTgq7KZuN+GQ1uVjuuAOm+4shO+gRA5HDaNcWD+7IQwxy9zOgIAnvYdlut6NFYc++q+2KzD6La/P1sfzb0MQj8dCGI06ya5IHDCgqcUg485EGxAPCv7tDv3mPXf89pFk41irfRvDbcrgiOAS1LTpgi6o8iLv/Sd7OlwGHAAqsKCSH4PI43SYz3PGeu9oedWAnXgYZ51+N5xJ3Iyu+yuCd7bJ5iwAtudNJoHDFLby/McswvtzwddudgeVqu6RsQ9x4K7s0esm+/PiOkjW5lyQ0E/uaP2R+z7Z9yvJwYhIwechueyA9YkmL6+Iv6i3HieMR1x5VfHDu3YKVMZxAQ9m8cxd2rIa2E9EMtgGA3Neu4tCGq4PQuQvBpCYkdsleFG60bhSa9dp385uLFqTIiNuQkfcMSVqlI1UKUPYu68ZvU/6KcnyszHZ6jBbb4czdBeO4/sg6Jd+F1jo+eU4AlUyv5IYeHcpR1I6Pthd0e3y0t+eRa42fcwqNu7PpDAWs/eeAyGXWXw1qIEERmFbZAZMoZUZcQB0gBq3w+5cZjat27hzNHukW7o2WaOrt5H9QSwMEFAAAAAgAAAAhALwmElImAwAAUQcAABoAAABjb25maWcvYnAuc2NoZW1hLnJ1bnMueWFtbI1VUU/bQAx+z6+waKVuU5oA2pDIEE9DEy/TtE17gU5yk0tzcLmE82Vdh/rf57tLQ4AK1pcmd/Zn+/NnZwJ5o0u5SpdtQnklakxMpynZYK2iSTSBn6hkgVYQsAU/jO5BahCYV7AUOq9qNLdQNqoQJmY/I6hTllJnnQ4G6ZnGWpynZ2Sxbs9TaAysG3NLLebiFduMUW0lgK1ASbKAo8BrIznHGNhF6pW3G2UllQBpYY0ENRYCStPUjIa62CES3xNzUbdoJDUaLC7ZyXksO6msd4E3RZNT+uni++XnL0ldxEAit5KtT94mjHfhyHDpSfK4gQ1OhCQHdSc+b1dWAcuND+nNGWmXS/CZEcP5uh3WuhL6SUm7UhIgkzNnSvk7R2HSboA7md8SM7Qjy1W7Qqk5vK0YMvQ6IDtmPH3QaJFE0TRcZnBQWdtSlqY3zMi8l0djVmlhsLTp8eHx4fzoOA0XB5GVVokMvu1CFoJyI1tHUAY/nncE99TjeY5HzFCgZl3Jnlt2VdS/50guYbtpOW6zvOFuREbcddKIIoOrAT72LY69PS0iLArpskL11TStMFYKyqBERYKrL0RJWQRAqlu5f4CAT9awtPxBi9YKw1XNfl3h/O/h/HTR/88X76azKGofcNlhyCPAvcbLThmibeZGKLTyt3Ahq0EkFVIFTenNWLDEcuIxtkLbZJRwT4g7GJHigGKPsPBXL5Hha31USl99lcH9mJV4TMluowyDvGyv/eq4Toaz6+Q5cdd+q0xn2z6Oy/GlOFTh8YeT7Iq9cV4u7k/eb4Oza/Z/UT0bqUxqT2djePw+hr3mBpWGnfN0mnmKnMzHjKMxuPHvvIUuraiZySP/3ml514n+yJoucCvD+/3UiJLHbZJ68aVOeQeuEK/XPZWcz3uK9pbkveJQAk+0E4qrIa9QKaFXzkRLPvd2fp7COsIe1BXrlIZAsu6c/hrNjNSt3QwLg3sD2q+LfdWHunq4Z2J8JEeXhB/PeBRt0du9ps196gysDbqRPBYr/iS5jjB+zQ3hZ/wTnk+3gxen8EInwu8hQzYV2iFcsRhQ8YenkbRZbKN/UEsBAhQDFAAAAAgAAAAhAJGXrPX5AQAAZAQAACwAAAAAAAAAAAAAAKSBAAAAAGNvbmZpZy9iZW5jaG1hcmsvYnAuZGF0YS5iZW5jaG1hcmsuY29yZS55YW1sUEsBAhQDFAAAAAgAAAAhAEY/0Ft6BQAAMA8AAB8AAAAAAAAAAAAAAKSBQwIAAGNvbmZpZy9icC5zY2hlbWEuYmVuY2htYXJrLnlhbWxQSwECFAMUAAAACAAAACEAZ72rp9ADAACuCAAAHQAAAAAAAAAAAAAApIH6BwAAY29uZmlnL2JwLnNjaGVtYS5jaXJjdWl0LnlhbWxQSwECFAMUAAAACAAAACEAZZQF+k0JAABcHgAAGwAAAAAAAAAAAAAApIEFDAAAY29uZmlnL2JwLnNjaGVtYS5ub2lzZS55YW1sUEsBAhQDFAAAAAgAAAAhAC8S4vJUBAAAjwoAAB8AAAAAAAAAAAAAAKSBixUAAGNvbmZpZy9icC5zY2hlbWEucHJvY2Vzc29yLnlhbWxQSwECFAMUAAAACAAAACEAfqqt8M8HAADJGAAAHAAAAAAAAAAAAAAApIEcGgAAY29uZmlnL2JwLnNjaGVtYS5yZXN1bHQueWFtbFBLAQIUAxQAAAAIAAAAIQC8JhJSJgMAAFEHAAAaAAAAAAAAAAAAAACkgSUiAABjb25maWcvYnAuc2NoZW1hLnJ1bnMueWFtbFBLBQYAAAAABwAHABoCAACDJQAAAAA="

In [ ]:
#@title Unpack the payloads, list each file's SHA-256, and define %%repo_file
import base64, hashlib, html, io, sys, tempfile, zipfile
from pathlib import Path

from IPython.core.magic import register_cell_magic
from IPython.display import HTML, display

WORK = Path(tempfile.mkdtemp(prefix="bellbench-"))
manifests = []
for title, payload in (("The code payload", CODE), ("The nine cases' runs", RESULTS),
                       ("The extended benchmarks' runs", STUDIES), ("The schemas", SCHEMAS)):
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
        archive.extractall(WORK)
        rows = "".join(f"<tr><td><code>{html.escape(n)}</code></td>"
                       f"<td><code>{hashlib.sha256(archive.read(n)).hexdigest()}</code></td></tr>"
                       for n in archive.namelist())
        manifests.append(f"<details><summary><b>{title}: {len(archive.namelist())} files, each with its SHA-256</b>"
                         f"</summary><table><tr><th>file</th><th>SHA-256</th></tr>{rows}</table></details>")
sys.path.insert(0, str(WORK / "src"))
from bellbench import configs  # reads every config file from WORK


@register_cell_magic
def repo_file(line, cell):
    """Write the cell's text to the repository path on the magic's line, inside WORK, and show its content hash."""
    path = WORK / line.strip()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(cell, encoding="utf-8")
    print(f"wrote {line.strip()} · content hash {configs.content_hash(path)[:19]}…")


display(HTML("".join(manifests)))
print(f"unpacked the four payloads into {WORK}")

**The schemas, and the core benchmark file:** the schemas say what a valid processor file, noise profile, circuit file, benchmark file, run list and run record must hold; the core benchmark file gives the workbench its default noise profile, shots and seed, and nothing in this notebook reports its runs. Each is folded: click one to read it.

In [ ]:
display(HTML("".join(
    f"<details><summary><b>{path.relative_to(WORK).as_posix()}</b> · content hash {configs.content_hash(path)[:19]}…"
    f"</summary><pre>{html.escape(path.read_text(encoding='utf-8'))}</pre></details>"
    for path in [*sorted((WORK / "config").glob("bp.schema.*.yaml")), WORK / "config/benchmark/bp.data.benchmark.core.yaml"])))

---

## 2. The task

**On each of three processor graphs, create the Bell pair |β₀₀⟩ = (|00⟩ + |11⟩)/√2 between two fixed qubits, measure how close it comes, and test one strategy that protects it.** Two qubits share a unit vector in ℂ⁴, and |β₀₀⟩ is the vector (1, 0, 0, 1)/√2. A two-qubit gate acts only between neighbours on the graph, so a pair made at one port must be moved, or routed, to the other, and each added gate adds noise.

We verify the pair by measuring the two ports in three bases and combining the three correlators into the fidelity:

$$F = \frac{1 + \langle XX\rangle - \langle YY\rangle + \langle ZZ\rangle}{4}$$

F = 1 is the target exactly; F above 1/2 shows entanglement. The challenge fixes nine cases, each graph under three conditions:

| Graph | Ideal | Noisy | Noisy, protected |
|---|---|---|---|
| 2016-inspired | Case 1 | Case 2 | Case 3 |
| Contemporary | Case 4 | Case 5 | Case 6 |
| Hyperbolic-inspired | Case 7 | Case 8 | Case 9 |

<details><summary><b>The linear algebra behind the formula</b></summary>

- A correlator ⟨BB⟩ is the expectation value of B ⊗ B in the pair's state ρ, Tr(ρ (B ⊗ B)), for B one of the Pauli matrices X = [[0, 1], [1, 0]], Y = [[0, −i], [i, 0]] and Z = [[1, 0], [0, −1]]. From shots it is P(00) + P(11) − P(01) − P(10).
- |β₀₀⟩⟨β₀₀| = (I⊗I + X⊗X − Y⊗Y + Z⊗Z)/4, so F = ⟨β₀₀|ρ|β₀₀⟩, the overlap of the prepared state with the target.
- ZZ sees a bit flip, XX a phase flip, and YY either; all three together tell |β₀₀⟩ from the other three Bell states, |β₀₁⟩, |β₁₀⟩ and |β₁₁⟩.
- A state that is not entangled cannot have F above 1/2, so 1/2 is the line a classical correlation cannot cross.

The full statement of the task, with its sources, is [CHALLENGE.md](https://github.com/Rickory2021/bell-transport-bench/blob/submission/docs/CHALLENGE.md).

</details>

---

## 3. Methods

### 3.1 The three processor graphs

**We use the organisers' three coupling graphs exactly, with the ports they fix, and explain results by three features of each graph:** the hops between the ports, the paths between the ports that share no qubit (disjoint paths), and the most edges at any qubit. The three processor files come first; the figure and the table are drawn from them.

In [ ]:
%%repo_file config/processor/bp.data.processor.ibm2016.yaml
# config/processor/bp.data.processor.ibm2016.yaml
#
# The 2016-inspired reference processor model, the organisers' graph (docs/DESIGN.md, section 3).
# Validated by config/bp.schema.processor.yaml.

name: ibm2016
label: 2016-inspired reference
reference: ideal-and-noisy-example
notes: >-
  The organisers' example notebook, section 1 (the models cell), gives the T graph
  as an edge list with its ports; the organisers stated at the event that the
  graphs are exact. Ids and positions are the notebook's: slide 4's drawings carry
  no qubit numbers, so the notebook labels the qubits by position.
qubits:
  - {id: 0, position: [0, 2]}
  - {id: 1, position: [-1.55, 0.75]}
  - {id: 2, position: [1.55, 0.75]}
  - {id: 3, position: [0, -0.45]}
  - {id: 4, position: [0, -2]}
ports: [0, 4]
edges:
  - [0, 1]
  - [0, 2]
  - [0, 3]
  - [3, 4]

In [ ]:
%%repo_file config/processor/bp.data.processor.contemporary.yaml
# config/processor/bp.data.processor.contemporary.yaml
#
# The Contemporary IBM-style processor model, the organisers' graph (docs/DESIGN.md, section 3).
# Validated by config/bp.schema.processor.yaml.

name: contemporary
label: Contemporary IBM-style
reference: ideal-and-noisy-example
notes: >-
  The organisers' example notebook, section 1 (the models cell), gives the 9-node graph
  as an edge list with its ports; the organisers stated at the event that the
  graphs are exact. Ids and positions are the notebook's: slide 4's drawings carry
  no qubit numbers, so the notebook labels the qubits by position.
qubits:
  - {id: 0, position: [-2.1, 1]}
  - {id: 1, position: [-0.9, 1.7]}
  - {id: 2, position: [0.3, 1]}
  - {id: 3, position: [1.5, 1.7]}
  - {id: 4, position: [-1.5, -0.35]}
  - {id: 5, position: [-0.3, 0.35]}
  - {id: 6, position: [0.9, -0.35]}
  - {id: 7, position: [2.1, 0.35]}
  - {id: 8, position: [-0.3, -1.65]}
ports: [0, 7]
edges:
  - [0, 1]
  - [0, 4]
  - [1, 2]
  - [1, 5]
  - [2, 3]
  - [2, 5]
  - [2, 6]
  - [3, 7]
  - [4, 5]
  - [5, 6]
  - [5, 8]
  - [6, 7]

In [ ]:
%%repo_file config/processor/bp.data.processor.hyperbolic.yaml
# config/processor/bp.data.processor.hyperbolic.yaml
#
# The Hyperbolic-inspired processor model, the organisers' graph (docs/DESIGN.md, section 3).
# Validated by config/bp.schema.processor.yaml.

name: hyperbolic
label: Hyperbolic-inspired model
reference: ideal-and-noisy-example
notes: >-
  The organisers' example notebook, section 1 (the models cell), gives the 13-node graph
  as an edge list with its ports; the organisers stated at the event that the
  graphs are exact. Ids and positions are the notebook's: slide 4's drawings carry
  no qubit numbers, so the notebook labels the qubits by position.
qubits:
  - {id: 0, position: [0, 2.08]}
  - {id: 1, position: [-2, 1.14]}
  - {id: 2, position: [0, 1.14]}
  - {id: 3, position: [2, 1.14]}
  - {id: 4, position: [-1, 0.59]}
  - {id: 5, position: [1, 0.59]}
  - {id: 6, position: [0, 0]}
  - {id: 7, position: [-1, -0.59]}
  - {id: 8, position: [1, -0.59]}
  - {id: 9, position: [-2, -1.14]}
  - {id: 10, position: [0, -1.14]}
  - {id: 11, position: [2, -1.14]}
  - {id: 12, position: [0, -2.08]}
ports: [0, 12]
edges:
  - [0, 1]
  - [0, 3]
  - [1, 9]
  - [3, 11]
  - [9, 12]
  - [11, 12]
  - [2, 4]
  - [2, 5]
  - [4, 7]
  - [5, 8]
  - [7, 10]
  - [8, 10]
  - [0, 2]
  - [1, 4]
  - [3, 5]
  - [7, 9]
  - [8, 11]
  - [10, 12]
  - [2, 6]
  - [4, 6]
  - [5, 6]
  - [6, 7]
  - [6, 8]
  - [6, 10]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from bellbench.figures import processor_figure
from bellbench.paths import graph_metrics

fig = processor_figure()
display(fig)
plt.close(fig)
pd.DataFrame(
    {p: {k: graph_metrics(p)[k] for k in ("port_distance", "disjoint_paths", "max_degree", "path")}
     for p in configs.CORE_PROCESSORS}
).T.rename(columns={"port_distance": "hops between ports", "disjoint_paths": "disjoint paths",
                   "max_degree": "most edges at a qubit", "path": "pair's path"})

### 3.2 The noise model

**Our noise model uses the calibration values of IBM's Boston processor, taken from the FakeBoston snapshot recorded on 2026-04-17.** Aer's own `NoiseModel.from_backend` builds the gate, decay and reset channels from a Target holding those values, and the model then replaces Aer's readout errors with the profile's, which differ in the two directions ([`src/bellbench/noise.py`](https://github.com/Rickory2021/bell-transport-bench/blob/submission/src/bellbench/noise.py)). Each value is the median over the snapshot's qubits and couplers, and every qubit and coupler of all three graphs takes it, so the graphs differ in connectivity alone. Nothing here ran on hardware.

- **Modelled:** gate errors, decay (T1 and T2) during gates and while idle, and readout errors that differ in the two directions. Circuits are scheduled before noise is applied, so a qubit waiting for a 2.2 µs measurement decays while it waits.
- **Set per run, by none of the nine cases:** a mid-circuit measurement error, for a circuit that reads a qubit and then uses it again. No case does; the earlier XX check on the 2016-inspired graph, which appendix A.1 runs for comparison, sets it to 0.01. One run per graph in appendix A also sets the readout error of the qubits it reads to 0, to test what readout costs.

In [ ]:
%%repo_file config/noise/bp.data.noise.boston-2026.yaml
# config/noise/bp.data.noise.boston-2026.yaml
#
# Medians of IBM's FakeBoston snapshot (Heron r3). Validated by
# config/bp.schema.noise.yaml; tests/test_config.py pins every value to the
# snapshot and every comment to the schema.

name: boston-2026
label: FakeBoston medians, calibrated 2026-04-17
kind: snapshot
reference: qiskit-fake-provider
snapshot:
  class: FakeBoston
  package: qiskit-ibm-runtime==0.51.0
  calibrated: "2026-04-17"
  properties: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/boston/props_boston.json
  configuration: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/boston/conf_boston.json
notes: >-
  Medians over the FakeBoston snapshot's 156 qubits and 352 directed CZ couplers, as
  qiskit-ibm-runtime 0.51.0 pins them. Every qubit and coupler of a processor
  graph takes these values.
# Time step in seconds: the hardware's clock tick. Null when the snapshot records none.
time_step: 4.0e-9
qubit:
  # T1 in seconds: how long a qubit holds 1 before relaxing to 0; it is still 1 after time t with probability exp(-t/T1).
  t1_relaxation_time: 292.43565062547934e-6
  # T2 in seconds: how long a qubit keeps the phase between 0 and 1; never more than 2 T1.
  t2_dephasing_time: 352.96959961159655e-6
  readout_error:
    # Readout error: the probability of reading 1 when the qubit was 0.
    reads_1_when_0: 0.00390625
    # Readout error: the probability of reading 0 when the qubit was 1.
    reads_0_when_1: 0.0048828125
# Native gates, by Qiskit name: the qubits each acts on (1 or 2), its error probability per use, and its duration in seconds.
gates:
  cz: {qubits: 2, error_probability: 0.0012747211439835399, duration: 68.0e-9}
  sx: {qubits: 1, error_probability: 0.00014470700669960387, duration: 32.0e-9}
  x: {qubits: 1, error_probability: 0.00014470700669960387, duration: 32.0e-9}
  rz: {qubits: 1, error_probability: 0.0, duration: 0.0}
# Measurement: its duration in seconds. Its errors are the readout errors.
measure: {duration: 2.2e-6}
# Reset to 0: its duration in seconds.
reset: {duration: 2.232e-6}
borrowed: []

In [ ]:
profile = configs.load_noise("boston-2026")
q, g = profile["qubit"], profile["gates"]
pd.DataFrame(
    [("T1", q["t1_relaxation_time"] * 1e6, "µs"), ("T2", q["t2_dephasing_time"] * 1e6, "µs"),
     ("reads 1 when 0", q["readout_error"]["reads_1_when_0"], ""), ("reads 0 when 1", q["readout_error"]["reads_0_when_1"], ""),
     ("CZ error", g["cz"]["error_probability"], ""), ("CZ duration", g["cz"]["duration"] * 1e9, "ns"),
     ("√X error", g["sx"]["error_probability"], ""), ("measurement", profile["measure"]["duration"] * 1e6, "µs")],
    columns=["quantity", "value", "unit"],
).set_index("quantity")

<details><summary><b>What the model leaves out</b></summary>

**The calibration snapshot gives one number per gate, per qubit's readout and per qubit's T1 and T2, and the model uses only those, so every effect below is left out.** Section 6 states what that means for the results; [DESIGN.md, section 4](https://github.com/Rickory2021/bell-transport-bench/blob/submission/docs/DESIGN.md#4-noise-model) holds the full noise model.

| Left out | What it is | Why it is left out |
|---|---|---|
| Qubits that start a shot excited | A qubit begins in \|1⟩ instead of \|0⟩ | A run can set it as a conditional channel, but the snapshot gives no value for it, and no run here sets it |
| Measurement-induced dephasing | Reading one qubit disturbs its neighbours' phase | The snapshot gives no value for it; every read in the nine cases is at the end of the circuit |
| Always-on ZZ coupling | Neighbours keep a small interaction while idle | The snapshot gives no value for it; an idle qubit here only decays |
| Spectator errors | A gate disturbs the qubits next to the ones it acts on | The snapshot gives each gate's error on its own qubits only |
| Coherent errors | Systematic over- or under-rotations that add up instead of averaging out | The snapshot gives each gate's error as one rate, with no angle, so the model applies it as a random error |
| Drift | Calibration values change over time | One snapshot is one moment; every run uses the same values, so the three graphs compare under the same noise |
| Feedforward latency | The delay before a circuit can act on a mid-circuit result | No case acts on a result during the circuit; every read is used after the shot |
| Cosmic-ray bursts | Rare events that raise error rates across a chip at once | They correlate errors across qubits and shots, which a per-gate, per-shot model cannot express |
| Leakage | A qubit leaving \|0⟩ and \|1⟩ for a higher level | The simulator treats each qubit as two-level, and the snapshot gives no value for it |
| Frequency crowding | Qubits with close frequencies disturbing each other | The snapshot gives no coupling between qubit frequencies |

</details>

### 3.3 The benchmark

**One benchmark file fills the nine cases: it names each run, a circuit file under the noise profile, the shots and the base seed, and which run fills each case.** Cases 1, 2, 4, 5, 7 and 8 are the organisers' baseline; cases 3, 6 and 9 are our protection, the same steps on every graph. The file runs nothing else. To separate what the transport buys from what the check buys, section 4.2 takes the transport without the check from each graph's extended benchmark (appendix A), which runs it under the same shots and seed. Each run takes 100,000 shots per basis. The notebook makes the benchmark itself at the end of section 3.5, once its circuit files are written.

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.deliverable-2026-11-17.yaml
# config/benchmark/bp.data.benchmark.deliverable-2026-11-17.yaml

name: deliverable-2026-11-17
label: GHZ disentangling with a vacancy read, the nine cases
notes: >-
  The challenge's nine cases (docs/CHALLENGE.md, The nine core cases) as the
  submission reports them, under boston-2026 as in the core benchmark, so the three
  graphs differ only in connectivity. Cases 1, 2, 4, 5, 7 and 8 are the organisers'
  baseline, exactly as the core benchmark runs it. Cases 3, 6 and 9 are one protection
  strategy, the same builder steps on every graph: ghz_uncompute makes a GHZ state
  along the pair's path, then disentangles the path's interior qubits with gates
  controlled from the first port, leaving them in |0>; vacancy then reads them at the
  end of the circuit, and postselection discards each shot where any reads 1. The read
  catches a bit flip on the interior qubits, and on the first port before the last
  gate, which copies it onto an interior qubit; it cannot see a phase flip. No qubit
  is read mid-circuit, so no case needs the mid-circuit measurement channel. The file runs only the runs that fill a case. Each graph's extended
  benchmark, deliverable-2026-11-17-extended-<processor>, runs the same protected case
  beside the alternatives under the same shots and seed, among them the transport
  without the read, so the submission separates what the transport buys from what the
  read buys. Every run takes 100,000 shots per basis, five times the core's 20,000, so
  F's standard error is about 0.0003 and a difference of 0.001 between two runs stands
  at more than two combined standard errors; the baseline cases therefore agree with
  the core's within their errors rather than repeating its numbers.
shots: 100000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  - {run: baseline-contemporary, circuit: baseline-contemporary, noise: boston-2026}
  - {run: baseline-hyperbolic, circuit: baseline-hyperbolic, noise: boston-2026}
  # The protection: the transport with its vacancy read at the end.
  - {run: ghz-uncompute-vacancy-ibm2016, circuit: ghz-uncompute-vacancy-ibm2016, noise: boston-2026}
  - {run: ghz-uncompute-vacancy-contemporary, circuit: ghz-uncompute-vacancy-contemporary, noise: boston-2026}
  - {run: ghz-uncompute-vacancy-hyperbolic, circuit: ghz-uncompute-vacancy-hyperbolic, noise: boston-2026}
cases:
  - {case: 1, run: baseline-ibm2016, simulation: ideal}
  - {case: 2, run: baseline-ibm2016, simulation: noisy}
  - {case: 3, run: ghz-uncompute-vacancy-ibm2016, simulation: noisy}
  - {case: 4, run: baseline-contemporary, simulation: ideal}
  - {case: 5, run: baseline-contemporary, simulation: noisy}
  - {case: 6, run: ghz-uncompute-vacancy-contemporary, simulation: noisy}
  - {case: 7, run: baseline-hyperbolic, simulation: ideal}
  - {case: 8, run: baseline-hyperbolic, simulation: noisy}
  - {case: 9, run: ghz-uncompute-vacancy-hyperbolic, simulation: noisy}

### 3.4 How a run works, in Qiskit

**Inside a run there is only Qiskit and Aer. This section makes case 3 by hand, from the files written above, with nothing but Qiskit and Aer, and checks each step against the same run made by the package.** The package wraps these steps in checks, records and views; everything after this section uses it. Case 3 is GHZ disentangling followed by a read of the emptied qubit on the 2016-inspired graph, section 3.5 explains both, and its circuit is short: three two-qubit gates on qubits 0, 3 and 4. First the helpers the rest of the notebook uses, then case 3's circuit file.

In [ ]:
#@title The notebook's helpers: make and compare the runs, draw the circuit views and charts
from html import escape

import numpy as np
import yaml
from bellbench.circuits import circuit_to_qasm
from bellbench.run import CircuitRun, comparison_rows, current_benchmark, read_table, run_benchmark
from bellbench.views.base import emit, fold
from bellbench.views.show import Compiled, basis_html, show_parts
from bellbench.workbench import Result, Workbench

REPORTED = "deliverable-2026-11-17"
NOISE, SHOTS, BASE_SEED = "boston-2026", 100000, 20261116  # every benchmark here shares them
MADE = WORK / "made"  # the runs this notebook makes; the recorded runs stay where the payloads put them
BASES = ("XX", "YY", "ZZ")


def make(benchmark):
    """Make every run the benchmark file lists, from the files written above, into a folder of this notebook's own."""
    folder = run_benchmark(benchmark, root=MADE)
    runs = yaml.safe_load((folder / "bp.data.runs.yaml").read_text())["runs"]
    print(f"made {len(runs)} runs of {benchmark}: {', '.join(runs)}")


def made(benchmark):
    """The folder of the benchmark as this notebook made it."""
    return current_benchmark(benchmark, root=MADE)


def recorded(benchmark):
    """The folder of the benchmark as the repository recorded it, from the payloads."""
    return current_benchmark(benchmark)


def circuit_views(circuit, folded=False):
    """A circuit file as the workbench shows it, compiled here with the benchmarks' noise profile and seed, without
    simulating: the graph and the written circuit, then the step-through folded, then one fold holding a fold per
    basis: its compiled circuit, its timelines and its OpenQASM 3. folded=True, for the appendix, keeps only the text lines open and folds the graph, the written
    circuit and the cost table too."""
    run = CircuitRun(circuit, NOISE, SHOTS, BASE_SEED).compile()
    view = Compiled.of_run(run)
    bases = "".join(fold(f"{basis}: the compiled circuit, its timeline and its OpenQASM 3",
                         basis_html(view, basis) + f"<pre>{escape(circuit_to_qasm(run.compiled[basis].compiled))}</pre>").content
                    for basis in BASES)
    parts = [*show_parts(view, step=True), fold("Compiled circuits, by basis", bases)]
    if folded:
        inside = [part for part in parts
                  if part.kind == "png" or (part.kind == "html" and not part.content.startswith("<details"))]
        lines = [part for part in parts if part.kind == "markdown"]
        rest = [part for part in parts if part not in inside and part not in lines]  # the folds, and any other kind
        pictures = "".join(f'<img style="max-width:100%" src="data:image/png;base64,{part.content}">'
                           if part.kind == "png" else part.content for part in inside)
        parts = [*lines, fold(f"{circuit}: the graph, the written circuit and its cost per basis", pictures), *rest]
    emit(parts)


SERIES = dict(zip(configs.CORE_PROCESSORS, ("#2a78d6", "#eb6834", "#1baf7a")))  # validated categorical slots 1 to 3
NAME = {"ibm2016": "2016-inspired", "contemporary": "contemporary", "hyperbolic": "hyperbolic-inspired"}
INK, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"


def chart(figsize, grid="y"):
    """A figure in the notebook's one chart style: a light surface, a recessive grid on one axis, no top or right
    spine. Every chart colours a graph the same way, by SERIES."""
    fig, ax = plt.subplots(figsize=figsize, facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    ax.grid(axis=grid, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)
    return fig, ax


def finish(fig, title, note, legend_ax=None):
    """Title top left, a muted note top right, and, for a chart of several graphs, a legend of them under the title."""
    if legend_ax is not None:
        legend_ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3, frameon=False, fontsize=9,
                         handletextpad=0.3, columnspacing=1.4, labelcolor=INK)
    fig.text(0.01, 0.985, title, fontsize=12, fontweight="bold", color=INK, va="top")
    fig.text(0.99, 0.985, note, fontsize=9, color=MUTED, va="top", ha="right")
    fig.tight_layout(rect=(0, 0, 1, 0.93 if legend_ax is not None else 0.95))
    display(fig)
    plt.close(fig)


def ladder(benchmark):
    """One graph's extended benchmark as a horizontal bar per run, in the order its benchmark file lists the runs: F of
    the kept shots with one standard error, labelled with its acceptance where the run discards shots. The protected
    case is the solid bar. The axis starts near the lowest F, not at 0, and its label says so."""
    t = read_table(made(benchmark) / "comparison.csv").query("simulation == 'noisy'").reset_index(drop=True)
    processor = t.processor.iloc[0]
    colour = SERIES[processor]
    y = np.arange(len(t))[::-1]  # the file's first run on top
    protected = [run == f"ghz-uncompute-vacancy-{processor}" for run in t.run]
    low = float(np.floor((t.fidelity - t.fidelity_se).min() / 0.002) * 0.002) - 0.002
    high = float((t.fidelity + t.fidelity_se).max())
    fig, ax = chart((8, 0.38 * len(t) + 1.4), grid="x")
    ax.barh(y, t.fidelity - low, left=low, height=0.62, color=colour, alpha=None, zorder=2,
            edgecolor=SURFACE, linewidth=2)
    for bar, p in zip(ax.patches, protected):
        bar.set_alpha(1 if p else 0.38)
    ax.errorbar(t.fidelity, y, xerr=t.fidelity_se, fmt="none", ecolor=INK, elinewidth=1, capsize=2, zorder=3)
    for yi, f, e, a, p in zip(y, t.fidelity, t.fidelity_se, t.acceptance, protected):
        ax.annotate(f"{f:.4f}" + ("" if a == 1 else f", kept {a:.3f}") + (" (protected case)" if p else ""), (f + e, yi),
                    xytext=(6, 0), textcoords="offset points", va="center", fontsize=8, color=INK if p else MUTED)
    ax.set_yticks(y, t.run)
    ax.tick_params(axis="y", colors=INK, length=0)
    ax.set_xlim(low, high + 0.35 * (high - low))  # room right for the labels
    ax.set_xlabel(f"F of the kept shots (± 1 SE); the axis starts at {low:.3f}, not 0", color=INK, fontsize=10)
    finish(fig, f"Every run on the {NAME[processor]} graph", benchmark)


def beside_record(benchmark, extra=()):
    """Each run this notebook made beside the same run as recorded: F and acceptance from each, whether the counts
    and the compiled circuits are identical, and whether the files the record names hash to the files written here.
    `extra` adds (made folder, recorded folder) pairs for runs made outside the benchmark. A different commit or
    package version is listed below the table, not counted as a difference."""
    pairs = [(made(benchmark) / run, recorded(benchmark) / run)
             for run in yaml.safe_load((made(benchmark) / "bp.data.runs.yaml").read_text())["runs"]]
    rows, provenance = [], set()
    for here, then in [*pairs, *extra]:
        mine, rec = (yaml.safe_load((f / "bp.data.result.yaml").read_text()) for f in (here, then))
        f_here, a_here = noisy_scores(here)
        f_then, a_then = noisy_scores(then)
        refs = [rec["circuit"], rec["noise_control"]["processor"], rec["noise_control"]["noise"]]
        rows.append({"run": then.name, "F here": f"{f_here:.4f}", "F recorded": f"{f_then:.4f}",
                     "acceptance here": f"{a_here:.3f}", "acceptance recorded": f"{a_then:.3f}",
                     "identical": (f_here, a_here) == (f_then, a_then) and mine["compiled"] == rec["compiled"]
                                  and (here / "counts.csv").read_text() == (then / "counts.csv").read_text(),
                     "same files": all(configs.content_hash(WORK / r["path"]) == r["hash"] for r in refs)})
        provenance |= {key for key in ("commit", "packages") if mine["provenance"][key] != rec["provenance"][key]}
    table = pd.DataFrame(rows).set_index("run")
    display(table)
    print("every run identical to its record, counts, circuits and numbers" if table.identical.all() and table["same files"].all()
          else "SOME RUNS DIFFER from their records: see the table")
    print("provenance that differs, not counted:", ", ".join(sorted(provenance)) or "none")


def noisy_scores(folder):
    """F of the kept shots and the acceptance of a run folder's noisy simulation, from its simulations table."""
    row = read_table(folder / "simulations.csv").query("simulation == 'noisy'").iloc[0]
    return float(row.fidelity), float(row.acceptance)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-ibm2016.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-ibm2016.yaml

name: ghz-uncompute-vacancy-ibm2016
label: ghz_uncompute then vacancy at the end, ibm2016
processor: ibm2016
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: ghz_uncompute();
  then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] check_vac;
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[0], q[3];
  check_vac[0] = measure q[3];

In [ ]:
package = CircuitRun("ghz-uncompute-vacancy-ibm2016", NOISE, SHOTS, BASE_SEED).run()  # the same run, by the package
print(f"the package's run of case 3: F = {package.scores['noisy'].accepted.value:.6f}, acceptance {package.scores['noisy'].acceptance:.6f}")

**1. The processor and its noise.** A Qiskit `Target` holds the native gates on each qubit and coupler with Boston's error rate and duration, each qubit's T1 and T2, and the measurement. Aer reads a `NoiseModel` from it: after each gate, a depolarising error and the relaxation for the gate's duration, and the same relaxation on every idle period. Readout error is then added per qubit, with its two directions kept apart.

In [ ]:
from qiskit import ClassicalRegister, qasm3
from qiskit.circuit.library import get_standard_gate_name_mapping
from qiskit.providers import BackendV2, Options
from qiskit.transpiler import InstructionProperties, PassManager, QubitProperties, Target, generate_preset_pass_manager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDelay
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError

noise = yaml.safe_load((WORK / f"config/noise/bp.data.noise.{NOISE}.yaml").read_text())
graph = yaml.safe_load((WORK / "config/processor/bp.data.processor.ibm2016.yaml").read_text())
n, qubit, readout = len(graph["qubits"]), noise["qubit"], noise["qubit"]["readout_error"]
couplers = [tuple(e) for e in graph["edges"]] + [tuple(reversed(e)) for e in graph["edges"]]  # a CZ either way
gate = get_standard_gate_name_mapping()

target = Target(num_qubits=n, dt=noise["time_step"],
                qubit_properties=[QubitProperties(t1=qubit["t1_relaxation_time"], t2=qubit["t2_dephasing_time"])] * n)
for name, spec in noise["gates"].items():
    where = couplers if spec["qubits"] == 2 else [(i,) for i in range(n)]
    target.add_instruction(gate[name], {w: InstructionProperties(error=spec["error_probability"], duration=spec["duration"])
                                        for w in where})
target.add_instruction(gate["measure"], {(i,): InstructionProperties(
    error=(readout["reads_1_when_0"] + readout["reads_0_when_1"]) / 2, duration=noise["measure"]["duration"]) for i in range(n)})
target.add_instruction(gate["reset"], {(i,): InstructionProperties(duration=noise["reset"]["duration"]) for i in range(n)})
target.add_instruction(gate["delay"], {(i,): None for i in range(n)})


class TargetOnly(BackendV2):
    """A backend that carries only the Target, so Aer can read a noise model from it."""

    def __init__(self, target):
        super().__init__(name="target-only")
        self._target = target

    target = property(lambda self: self._target)
    max_circuits = property(lambda self: None)
    _default_options = classmethod(lambda cls: Options())

    def run(self, *args, **kwargs):
        raise NotImplementedError("this backend only carries a Target")


noise_model = NoiseModel.from_backend(TargetOnly(target), readout_error=False)  # gate errors and relaxation, idle included
p10, p01 = readout["reads_1_when_0"], readout["reads_0_when_1"]
for i in range(n):
    noise_model.add_readout_error(ReadoutError([[1 - p10, p10], [p01, 1 - p01]]), [i])
print(noise_model)
print("the same noise model as the package's:", noise_model == package.nc.noise_model)

**2. The three circuits, compiled and scheduled.** The written circuit gets each basis's rotations on the ports and a measurement into a register named `port`. Qiskit's preset pass manager compiles each at optimisation level 1 into the Target, on the qubits as written, with the run's transpiler seed, which the package derives from the base seed and the run's settings. The scheduler then places every gate as late as possible and fills each idle period with a timed delay, which the noise model turns into relaxation.

In [ ]:
written = qasm3.loads(yaml.safe_load((WORK / "config/circuit/bp.data.circuit.ghz-uncompute-vacancy-ibm2016.yaml").read_text())["circuit"])
a, b = graph["ports"]


def gates_of(circuit):
    """Each instruction as (name, physical qubits, parameters), to compare two circuits whatever their register names."""
    return [(i.operation.name, tuple(circuit.find_bit(q).index for q in i.qubits), tuple(map(float, i.operation.params)))
            for i in circuit.data]


scheduled = {}
for basis in BASES:
    circuit = written.copy()
    port = ClassicalRegister(2, "port")
    circuit.add_register(port)
    for q in (a, b):
        if basis == "YY":
            circuit.sdg(q)
        if basis in ("XX", "YY"):
            circuit.h(q)
    circuit.measure([a, b], [port[0], port[1]])
    compiled = generate_preset_pass_manager(optimization_level=1, target=target, initial_layout=list(range(n)),
                                            seed_transpiler=package.seeds["transpile"]).run(circuit)
    scheduled[basis] = PassManager([ALAPScheduleAnalysis(target=target), PadDelay(target=target)]).run(compiled)
    print(f"{basis}: {dict(compiled.count_ops())}; the same gates on the same qubits as the package's:",
          gates_of(compiled) == gates_of(package.compiled[basis].compiled))

**3. Aer, then F.** One Aer job runs the three scheduled circuits with the run's noisy seed. Each count's key holds one bit string per register, the `port` register first. A shot is kept when the read of the emptied qubit gave 0, and each kept shot adds +1 to its basis's correlator when the two port bits agree and −1 when they differ; F = (1 + ⟨XX⟩ − ⟨YY⟩ + ⟨ZZ⟩)/4, as in section 2.

In [ ]:
counts = AerSimulator(noise_model=noise_model, seed_simulator=package.seeds["noisy"]).run(
    [scheduled[basis] for basis in BASES], shots=SHOTS).result().get_counts()


def correlator(counts):
    """The basis's correlator over the kept shots, and how many shots were kept."""
    kept = {}
    for key, k in counts.items():
        ports, *checks = key.split()
        if set("".join(checks)) <= {"0"}:
            kept[ports] = kept.get(ports, 0) + k
    total = sum(kept.values())
    return sum((1 if bits[0] == bits[1] else -1) * k for bits, k in kept.items()) / total, total


(xx, kx), (yy, ky), (zz, kz) = (correlator(c) for c in counts)
f_hand, acceptance_hand = (1 + xx - yy + zz) / 4, (kx + ky + kz) / (3 * SHOTS)
f_package, acceptance_package = package.scores["noisy"].accepted.value, package.scores["noisy"].acceptance
print(f"by hand:     F = {f_hand:.6f}, acceptance {acceptance_hand:.6f}")
print(f"the package: F = {f_package:.6f}, acceptance {acceptance_package:.6f}")
print("the same numbers" if (f_hand, acceptance_hand) == (f_package, acceptance_package) else "DIFFERENT numbers")

### 3.5 The circuits, written and compiled

**Each run is one circuit file: an OpenQASM 3 circuit on the graph's physical qubits, and how its measured bits are read.** One transport and one check, besides the baseline, fill the nine cases:

- **Baseline (cases 1, 2, 4, 5, 7, 8):** the organisers' recipe, H and CX at one port, SWAPs out to the other port and back, each SWAP three CX gates.
- **GHZ disentangling, our transport:** H on the first port, then a chain of CX gates along the path copies its bit onto every qubit, giving the GHZ state (|0…0⟩ + |1…1⟩)/√2 on the whole path. A second chain of CX gates walks back towards the first port, each gate controlled by the qubit before it, and returns every interior qubit to |0⟩, so only the two ports keep the shared bit: the Bell pair. The last of these gates is controlled by the first port. Liao et al. build a long-range CNOT the same way, preparing a GHZ state and disentangling it with gates ([arXiv:2411.14638](https://arxiv.org/abs/2411.14638)).
- **Vacancy read (cases 3, 6, 9):** after the transport, each interior qubit is measured in the Z basis at the end of the circuit. Each should read 0, so a shot where any reads 1 is discarded. A bit flip on an interior qubit reads 1, and so does a bit flip on the first port before the last gate, which copies it onto an interior qubit. The read cannot see a phase flip, and it adds no gate and no qubit.

For each graph, the two circuits its cases use, each as the repository holds it, then as the workbench shows it: the graph with the couplers the circuit uses, the written circuit and its cost, then two dropdowns: a step-through of the written circuit with no noise, and the compiled circuits, with a dropdown for each basis holding its compiled circuit, a timeline of when each qubit is busy, and the circuit as OpenQASM 3 (challenge material 02). Each view compiles its circuit here, as a run does, at optimisation level 1 into Boston's native gates (CZ, √X, X and Rz), without simulating.

#### The 2016-inspired graph: cases 1 to 3

In [ ]:
%%repo_file config/circuit/bp.data.circuit.baseline-ibm2016.yaml
# config/circuit/bp.data.circuit.baseline-ibm2016.yaml

name: baseline-ibm2016
label: Baseline, 2016-inspired reference
processor: ibm2016
analysis:
  name: postselect
notes: >-
  The baseline transport (swap_chain in src/bellbench/circuits.py), the organisers'
  recipe: H on the first port, explicit SWAPs out along the path [0, 3, 4], the lexicographically
  smallest shortest path between the ports, CX onto the second port, and the same
  SWAPs back, so the pair ends on the ports. No checks, so postselection keeps every
  shot.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[5] q;
  h q[0];
  swap q[0], q[3];
  cx q[3], q[4];
  swap q[0], q[3];

In [ ]:
circuit_views("baseline-ibm2016")

The protected case's circuit file is written out in section 3.4.

In [ ]:
circuit_views("ghz-uncompute-vacancy-ibm2016")

#### The contemporary graph: cases 4 to 6

In [ ]:
%%repo_file config/circuit/bp.data.circuit.baseline-contemporary.yaml
# config/circuit/bp.data.circuit.baseline-contemporary.yaml

name: baseline-contemporary
label: Baseline, Contemporary IBM-style
processor: contemporary
analysis:
  name: postselect
notes: >-
  The baseline transport (swap_chain in src/bellbench/circuits.py), the organisers'
  recipe: H on the first port, explicit SWAPs out along the path [0, 1, 2, 3, 7],
  the lexicographically smallest shortest path between the ports, CX onto the second
  port, and the same SWAPs back, so the pair ends on the ports. No checks, so postselection
  keeps every shot.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[9] q;
  h q[0];
  swap q[0], q[1];
  swap q[1], q[2];
  swap q[2], q[3];
  cx q[3], q[7];
  swap q[2], q[3];
  swap q[1], q[2];
  swap q[0], q[1];

In [ ]:
circuit_views("baseline-contemporary")

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-contemporary.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-contemporary.yaml

name: ghz-uncompute-vacancy-contemporary
label: ghz_uncompute then vacancy at the end, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
    - - check_vac[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  ghz_uncompute(); then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_vac;
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[3];
  cx q[3], q[7];
  cx q[2], q[3];
  cx q[1], q[2];
  cx q[0], q[1];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[2];
  check_vac[2] = measure q[3];

In [ ]:
circuit_views("ghz-uncompute-vacancy-contemporary")

#### The hyperbolic-inspired graph: cases 7 to 9

In [ ]:
%%repo_file config/circuit/bp.data.circuit.baseline-hyperbolic.yaml
# config/circuit/bp.data.circuit.baseline-hyperbolic.yaml

name: baseline-hyperbolic
label: Baseline, Hyperbolic-inspired model
processor: hyperbolic
analysis:
  name: postselect
notes: >-
  The baseline transport (swap_chain in src/bellbench/circuits.py), the organisers'
  recipe: H on the first port, explicit SWAPs out along the path [0, 1, 9, 12], the
  lexicographically smallest shortest path between the ports, CX onto the second port,
  and the same SWAPs back, so the pair ends on the ports. No checks, so postselection
  keeps every shot.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[13] q;
  h q[0];
  swap q[0], q[1];
  swap q[1], q[9];
  cx q[9], q[12];
  swap q[1], q[9];
  swap q[0], q[1];

In [ ]:
circuit_views("baseline-hyperbolic")

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-hyperbolic.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-hyperbolic.yaml

name: ghz-uncompute-vacancy-hyperbolic
label: ghz_uncompute then vacancy at the end, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: ghz_uncompute();
  then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_vac;
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[12];
  cx q[1], q[9];
  cx q[0], q[1];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[9];

In [ ]:
circuit_views("ghz-uncompute-vacancy-hyperbolic")

**Every file the nine cases use is now written, so the notebook makes the benchmark itself:** each run compiled and simulated ideal and noisy, 100,000 shots per basis, with seeds derived from the base seed, into a folder of its own. Every table and chart from here on reads these runs; section 4.4 sets each beside the run the repository recorded.

In [ ]:
make(REPORTED)

### 3.6 What each run reports

**Each case reports F with its standard error, the acceptance (the fraction of shots kept), and the circuit's cost,** as the challenge's five measures require. Each run takes 100,000 shots per basis, simulated ideal and noisy with seeds derived from one base seed, 20261116, so every run repeats exactly.

- **F after postselection** is F of the kept shots; **F raw** is F of every shot. A protected result is never shown without its acceptance.
- **Standard error:** a correlator c from N shots has variance (1 − c²)/N; the three bases are separate shots, so SE(F) = √(Σ var)/4, with N the kept shots. Two runs differ by more than chance when their difference exceeds about two combined standard errors, √(SE₁² + SE₂²); this notebook gives each difference in those units.
- **Cost:** two-qubit gates, depth, SWAPs and qubits used, from the compiled circuit before scheduling, and the scheduled duration.
- **Two-qubit gates per accepted pair:** the gates divided by the acceptance, what each kept pair costs.
- **Infidelity removed per rejection:** (F − F raw)/(1 − p), how much a check lowers 1 − F for each unit of shots it discards.

### 3.7 Why we wrote the transports, rather than leave routing to the transpiler

**Qiskit's transpiler solves a different problem: it routes a circuit and tracks where its qubits end up, rather than bringing the pair back to the ports.** The cell gives the bare task, H on the first port then CX between the ports, to `generate_preset_pass_manager` at levels 1 to 3 with the graph's coupling map, CX as the two-qubit gate and the identity layout, over 20 seeds each, and counts the CX gates and how often the pair ends on the ports.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

rows = []
for processor in configs.CORE_PROCESSORS:
    proc = configs.load_processor(processor)
    edges = list(configs.processor_graph(proc).edges)
    coupling = CouplingMap([list(e) for e in edges] + [[b, a] for a, b in edges])
    a, b = proc["ports"]
    task = QuantumCircuit(len(proc["qubits"]))
    task.h(a)
    task.cx(a, b)
    for level in (1, 2, 3):
        cx, on_ports = set(), 0
        for seed in range(20):
            pm = generate_preset_pass_manager(level, coupling_map=coupling, basis_gates=["cx", "h", "rz", "sx", "x"],
                                              initial_layout=list(range(task.num_qubits)), seed_transpiler=seed)
            out = pm.run(task)
            cx.add(out.count_ops().get("cx", 0))
            final = out.layout.final_index_layout()
            on_ports += final[a] == a and final[b] == b
        rows.append({"graph": processor, "level": level, "CX gates": sorted(cx), "pair on the ports": f"{on_ports} of 20"})
made_cases = read_table(made(REPORTED) / "comparison.csv").query("simulation == 'noisy' and case in (3, 6, 9)")
ours = dict(zip(made_cases.processor, made_cases.two_qubit_gates.astype(int)))  # the read adds no gate
display(pd.DataFrame(rows).set_index(["graph", "level"]))
print("GHZ disentangling, pair on the ports every time:", ours)

---

## 4. Results

### 4.1 The nine cases

**Every ideal case gives F = 1 with every shot kept, and the protected cases keep 97.7% to 99.3% of their shots.** One noise profile, one base seed, 100,000 shots per basis; each ± is one standard error.

| Graph | Ideal | Noisy baseline | Protected: F after postselection, acceptance |
|---|---|---|---|
| 2016-inspired | 1 (case 1) | 0.9766 ± 0.0003 (case 2) | 0.9842 ± 0.0003, 0.993 (case 3) |
| Contemporary | 1 (case 4) | 0.9600 ± 0.0004 (case 5) | 0.9817 ± 0.0003, 0.977 (case 6) |
| Hyperbolic-inspired | 1 (case 7) | 0.9686 ± 0.0004 (case 8) | 0.9832 ± 0.0003, 0.985 (case 9) |

The table below gives every case the five measures the challenge asks every result to report, from the runs this notebook made of the deliverable-2026-11-17 benchmark, the benchmark that fills the nine cases.

In [ ]:
#@title Build the nine cases' table and charts
table_all = read_table(made(REPORTED) / "comparison.csv")
cases = table_all.dropna(subset=["case"]).sort_values("case")
raw = sum(cases[f"shots_raw_{b}"] for b in BASES)
kept = sum(cases[f"shots_accepted_{b}"] for b in BASES)
CONDITION = {1: "ideal", 2: "noisy baseline", 0: "noisy, protected"}
table = pd.DataFrame({
    "case": cases.case.astype(int),
    "graph": cases.processor,
    "condition": [CONDITION[int(c) % 3] for c in cases.case],
    "F ± SE": [f"{f:.4f} ± {e:.4f}" for f, e in zip(cases.fidelity, cases.fidelity_se)],
    "F raw": cases.fidelity_raw.round(4),
    "acceptance (95% interval)": [f"{a:.3f} ({lo:.3f} to {hi:.3f})"
                                  for a, lo, hi in zip(cases.acceptance, cases.acceptance_low, cases.acceptance_high)],
    "discarded shots": (raw - kept).astype(int),
    "two-qubit gates": cases.two_qubit_gates,
    "depth": cases.depth,
    "SWAPs": cases.swaps,
    "qubits used": cases.qubits_used,
    "schedule (µs)": (cases.scheduled_duration * 1e6).round(2),
}).set_index("case")
display(table)

CONDITIONS = ("ideal", "noisy baseline", "protected")
fig, ax = chart((8, 4.6))
ends = []
for processor, colour in SERIES.items():
    rows = cases[cases.processor == processor].sort_values("case")  # ideal, noisy baseline, protected
    x, y = np.arange(3), rows.fidelity.to_numpy()
    ax.plot(x, y, color=colour, lw=2, zorder=2)
    ax.errorbar(x, y, yerr=rows.fidelity_se, fmt="none", ecolor=colour, elinewidth=1, zorder=2)
    ax.scatter(x, y, s=70, color=colour, edgecolor=SURFACE, linewidth=2, zorder=3, label=NAME[processor])
    ax.annotate(f"{y[1]:.4f}", (1, y[1]), xytext=(9, -2), textcoords="offset points", va="top", fontsize=8, color=MUTED)
    ends.append((y[2], f"{NAME[processor]}  {y[2]:.4f}, kept {rows.acceptance.iloc[2]:.3f}"))
# The protected cases sit close together, so their labels are spread to at least GAP apart, keeping their order.
GAP = 0.0024
heights = [f for f, _ in sorted(ends)]
for i in range(1, len(heights)):
    heights[i] = max(heights[i], heights[i - 1] + GAP)
shift = (np.mean(heights) - np.mean([f for f, _ in ends]))
for h, (f, text) in zip(heights, sorted(ends)):
    ax.annotate(text, (2, f), xytext=(2.12, h - shift), textcoords="data", va="center", fontsize=8, color=INK,
                arrowprops=dict(arrowstyle="-", color=AXIS, lw=0.8, shrinkA=0, shrinkB=5))
ideal = cases[cases.simulation == "ideal"].fidelity
ax.annotate(f"F = {ideal.iloc[0]:g} on every graph" if ideal.nunique() == 1 else "", (0, ideal.max()), xytext=(10, 4),
            textcoords="offset points", ha="left", va="bottom", fontsize=8, color=MUTED)
ax.set_xticks(range(3), [f"{name}\n(cases {', '.join(str(int(c)) for c in sorted(cases.case) if int(c) % 3 == (i + 1) % 3)})"
                         for i, name in enumerate(CONDITIONS)])
ax.tick_params(axis="x", colors=INK)
ax.set_xlim(-0.2, 3.1)  # room right of the protected cases for their labels
ax.set_ylim(float(cases.fidelity.min()) - 0.006, 1.006)
ax.set_ylabel("F of the kept shots (± 1 SE)", color=INK, fontsize=10)
finish(fig, "The nine cases", f"{REPORTED}; every ± is smaller than its dot", legend_ax=ax)

### 4.2 What protection buys, and what it costs

**On every graph the protected case beats the noisy baseline on F while using fewer two-qubit gates, no SWAP and no extra qubit; its only added cost is the shots the read discards.** For each graph, a table sets the noisy baseline, the transport alone (made here with the nine cases' shots and seed, so the same seeds as its run in appendix A) and the protected case side by side on the challenge's five measures, then the workbench draws them: a table and a bar per run of its F, and a second bar for a run's kept shots, with its acceptance. The baseline's and the protected case's circuits are in section 3.5; each graph's block below writes out the transport alone, GHZ disentangling without the read, and makes it. Each difference is given with how many combined standard errors it spans.

In [ ]:
wb = Workbench(noise=NOISE, shots=SHOTS, base_seed=BASE_SEED)  # the nine cases' settings
ALONE = {}  # the transport alone on each graph, as made here


def side_by_side(processor):
    """Make the transport alone on one graph, keep its run beside the others, and compare it with the noisy baseline
    and the protected case as the workbench does."""
    baseline, protected = {"ibm2016": ("baseline-ibm2016", "ghz-uncompute-vacancy-ibm2016"), "contemporary": ("baseline-contemporary", "ghz-uncompute-vacancy-contemporary"), "hyperbolic": ("baseline-hyperbolic", "ghz-uncompute-vacancy-hyperbolic")}[processor]
    ALONE[processor] = wb.run(f"ghz-uncompute-{processor}")
    ALONE[processor].run.write(folder=MADE / "transport-alone" / f"ghz-uncompute-{processor}")
    here = made(REPORTED)
    return wb.compare(Result.from_folder(here / baseline), ALONE[processor], Result.from_folder(here / protected))

#### The 2016-inspired graph

**The protected case raised F by +0.0075 over the noisy baseline (17.1 combined standard errors), discarding 0.7% of shots, with 4 fewer two-qubit gates.** The transport alone gives +0.0052 of that, by dropping the baseline's SWAPs out and back; the read adds +0.0023 (5.7 SE), at no cost in gates or qubits.


| Measure | Noisy baseline (case 2) | Transport alone (appendix A) | Protected (case 3) | Protected minus baseline |
|---|---|---|---|---|
| F ± standard error | 0.9766 ± 0.0003 | 0.9818 ± 0.0003 | 0.9842 ± 0.0003 | +0.0075 (17.1 SE) |
| F raw, every shot | 0.9766 | 0.9818 | 0.9816 | +0.0050 |
| Acceptance | 1.000 | 1.000 | 0.993 | -0.007 |
| Discarded shots, of 300,000 | 0 | 0 | 2,172 | +2,172 |
| Two-qubit gates | 7 | 3 | 3 | -4 |
| Depth | 15 | 15 | 15 | +0 |
| SWAPs | 2 | 0 | 0 | -2 |
| Physical qubits, ancillas included | 3 | 3 | 3 | +0 |
| Schedule (µs) | 2.87 | 2.50 | 2.53 | -0.34 |

The transport alone:

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-ibm2016.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-ibm2016.yaml

name: ghz-uncompute-ibm2016
label: ghz_uncompute, ibm2016
processor: ibm2016
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: ghz_uncompute().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[0], q[3];

In [ ]:
circuit_views("ghz-uncompute-ibm2016")

In [ ]:
side_by_side("ibm2016")

#### The contemporary graph

**The protected case raised F by +0.0217 over the noisy baseline (40.5 combined standard errors), discarding 2.3% of shots, with 12 fewer two-qubit gates.** The transport alone gives +0.0145 of that, by dropping the baseline's SWAPs out and back; the read adds +0.0073 (15.5 SE), at no cost in gates or qubits.


| Measure | Noisy baseline (case 5) | Transport alone (appendix A) | Protected (case 6) | Protected minus baseline |
|---|---|---|---|---|
| F ± standard error | 0.9600 ± 0.0004 | 0.9745 ± 0.0004 | 0.9817 ± 0.0003 | +0.0217 (40.5 SE) |
| F raw, every shot | 0.9600 | 0.9745 | 0.9744 | +0.0144 |
| Acceptance | 1.000 | 1.000 | 0.977 | -0.023 |
| Discarded shots, of 300,000 | 0 | 0 | 6,789 | +6,789 |
| Two-qubit gates | 19 | 7 | 7 | -12 |
| Depth | 39 | 31 | 31 | -8 |
| SWAPs | 6 | 0 | 0 | -6 |
| Physical qubits, ancillas included | 5 | 5 | 5 | +0 |
| Schedule (µs) | 4.07 | 2.90 | 2.93 | -1.14 |

The transport alone:

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-contemporary.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-contemporary.yaml

name: ghz-uncompute-contemporary
label: ghz_uncompute, contemporary
processor: contemporary
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  ghz_uncompute().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[3];
  cx q[3], q[7];
  cx q[2], q[3];
  cx q[1], q[2];
  cx q[0], q[1];

In [ ]:
circuit_views("ghz-uncompute-contemporary")

In [ ]:
side_by_side("contemporary")

#### The hyperbolic-inspired graph

**The protected case raised F by +0.0146 over the noisy baseline (29.8 combined standard errors), discarding 1.5% of shots, with 8 fewer two-qubit gates.** The transport alone gives +0.0096 of that, by dropping the baseline's SWAPs out and back; the read adds +0.0049 (11.2 SE), at no cost in gates or qubits.


| Measure | Noisy baseline (case 8) | Transport alone (appendix A) | Protected (case 9) | Protected minus baseline |
|---|---|---|---|---|
| F ± standard error | 0.9686 ± 0.0004 | 0.9782 ± 0.0003 | 0.9832 ± 0.0003 | +0.0146 (29.8 SE) |
| F raw, every shot | 0.9686 | 0.9782 | 0.9784 | +0.0098 |
| Acceptance | 1.000 | 1.000 | 0.985 | -0.015 |
| Discarded shots, of 300,000 | 0 | 0 | 4,354 | +4,354 |
| Two-qubit gates | 13 | 5 | 5 | -8 |
| Depth | 27 | 23 | 23 | -4 |
| SWAPs | 4 | 0 | 0 | -4 |
| Physical qubits, ancillas included | 4 | 4 | 4 | +0 |
| Schedule (µs) | 3.47 | 2.70 | 2.73 | -0.74 |

The transport alone:

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-hyperbolic.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-hyperbolic.yaml

name: ghz-uncompute-hyperbolic
label: ghz_uncompute, hyperbolic
processor: hyperbolic
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: ghz_uncompute().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[12];
  cx q[1], q[9];
  cx q[0], q[1];

In [ ]:
circuit_views("ghz-uncompute-hyperbolic")

In [ ]:
side_by_side("hyperbolic")

**Across the three graphs,** F of the kept shots against what each kept pair costs in two-qubit gates, the gates divided by the acceptance (section 3.6). Each line runs from a graph's noisy baseline to its transport alone and its protected case: the transport moves left, by dropping SWAPs, and up; the read moves almost straight up, since it adds no gate and discards few shots.

In [ ]:
#@title Draw the chart of what the transport and the read each buy, on each graph
from matplotlib.ticker import FixedLocator, NullLocator

noisy = read_table(made(REPORTED) / "comparison.csv").query("simulation == 'noisy'")
alone = pd.DataFrame(comparison_rows({f"ghz-uncompute-{p}": r.run for p, r in ALONE.items()}, {})).query("simulation == 'noisy'")
noisy = pd.concat([noisy, alone.assign(case=np.nan)])
noisy["stage"] = [1 if pd.isna(c) else 0 if int(c) % 3 == 2 else 2 for c in noisy.case]
STAGE = ("noisy baseline", "transport alone", "+ vacancy read")
# Where each label sits beside its point, in points: (dx, dy, ha, va), by stage, then any one point's own placement.
PLACE = {0: (0, -12, "center", "top"), 1: (-9, 0, "right", "center"), 2: (9, 0, "left", "center"),
         ("ibm2016", 0): (9, 3, "left", "bottom")}
fig, ax = chart((8, 4.6))
for processor, colour in SERIES.items():
    rows = noisy[noisy.processor == processor].sort_values("stage")
    x, y = rows.two_qubit_gates_per_accepted_pair, rows.fidelity
    ax.plot(x, y, color=colour, lw=2, zorder=2)
    ax.errorbar(x, y, yerr=rows.fidelity_se, fmt="none", ecolor=colour, elinewidth=1, capsize=0, zorder=2)
    ax.scatter(x, y, s=70, color=colour, edgecolor=SURFACE, linewidth=2, zorder=3, label=NAME[processor])
    for (xi, yi, stage) in zip(x, y, rows.stage):
        dx, dy, ha, va = PLACE.get((processor, stage), PLACE[stage])
        text = f"{NAME[processor]}\n{STAGE[0]}" if stage == 0 else STAGE[stage]
        ax.annotate(text, (xi, yi), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, fontsize=8,
                    color=INK if stage == 0 else MUTED)
ax.set_xscale("log")
ax.xaxis.set_major_locator(FixedLocator([3, 5, 7, 10, 13, 19]))
ax.xaxis.set_minor_locator(NullLocator())
ax.xaxis.set_major_formatter(lambda v, _: f"{v:g}")
ax.set_xlim(1.9, 26)  # room left of the cheapest points for their labels
ax.set_ylim(float(noisy.fidelity.min()) - 0.006, float(noisy.fidelity.max()) + 0.003)
ax.set_xlabel("Two-qubit gates per kept pair (log scale)", color=INK, fontsize=10)
ax.set_ylabel("F of the kept shots (± 1 SE)", color=INK, fontsize=10)
finish(fig, "Quality against cost", "noisy cases and the transport alone, as made here", legend_ax=ax)

### 4.3 What the graphs explain

**All three graphs share one noise model, so their differences come from connectivity alone. Port distance sets the cost of moving the pair, protected or not; a spare path, a second path between the ports, lets a check add a little more F at a cost in shots, gates and qubits.**

| Graph | Hops between ports | Disjoint paths | Baseline gates, F | Transport alone gates, F | Protected F, acceptance | Plus an XX check on a spare path: F, acceptance |
|---|---|---|---|---|---|---|
| 2016-inspired | 2 | 1 | 7, 0.9766 | 3, 0.9818 | 0.9842, 0.993 | no spare path |
| Hyperbolic-inspired | 3 | 3 | 13, 0.9686 | 5, 0.9782 | 0.9832, 0.985 | 0.9845, 0.971 |
| Contemporary | 4 | 2 | 19, 0.9600 | 7, 0.9745 | 0.9817, 0.977 | 0.9827, 0.956 |

- **F falls as the ports grow apart, with or without protection:** the baseline, the transport alone and the protected case all rank the graphs by hops, 2, then 3, then 4. The protected cases stand 2.5 and 3.4 combined standard errors apart.
- **The 2016-inspired graph, with one path between its ports, gives the highest protected F.** The vacancy read uses the pair's own path at the end of the circuit, so it needs no spare path.
- **What a spare path buys:** on the hyperbolic-inspired graph, an XX check on a spare path after the read raised F by +0.0013 (3.3 SE), discarding 1.5% more of the shots, with 3 more two-qubit gates and 2 more qubits. On the contemporary graph it raised F by +0.0010 (2.2 SE), through the graph's five-edge qubit, whose extra error the model leaves out. The read alone gains 0.34 in F per discarded shot on the hyperbolic-inspired graph, the check 0.09 per extra discarded shot, so we report the read alone on every graph, which also keeps the three graphs' cases one strategy.
- **The hyperbolic-inspired graph is a schematic,** a 13-qubit graph of short distances and high degree, not a patch of a hyperbolic lattice; we claim no advantage from its geometry beyond these three features.

### 4.4 The notebook's runs beside the recorded ones

**Every number above comes from runs this notebook made; this table sets each beside the run the repository recorded.** One row per run of the nine cases and per transport alone from 4.2: F and acceptance as made here and as recorded, whether the counts, the compiled circuits and the numbers are identical, and whether the files the record names hash to the files this notebook wrote. Outside the team's git repository the commit cannot match, and with other package versions the numbers may not, which is why section 1 installs the pinned versions; both are listed below the table, not counted as a difference.

In [ ]:
beside_record(REPORTED, extra=[(MADE / "transport-alone" / f"ghz-uncompute-{p}", recorded(name) / f"ghz-uncompute-{p}")
                               for p, name in {"ibm2016": "deliverable-2026-11-17-extended-ibm2016", "contemporary": "deliverable-2026-11-17-extended-contemporary", "hyperbolic": "deliverable-2026-11-17-extended-hyperbolic"}.items()])

---

## 5. Interpretation

**What the results show, against the three things the challenge asks a comparison to explain:**

- **Fidelity against the operations compilation added.** Under one noise model, F tracks the two-qubit gates and the idle time a circuit adds. The baseline's SWAPs out and back cost 7, 13 and 19 two-qubit gates; GHZ disentangling's 3, 5 and 7 recover 22%, 31% and 36% of the fidelity the baseline lost, measured against F = 1.
- **Whether protection pays.** A read of the emptied path at the end of the circuit raised F above the same transport by more than two combined standard errors on every graph, discarding 0.7% to 2.3% of shots and adding no gate. The same read after the move chain gained less (appendix A).
- **Connectivity against noise.** Every graph runs under the same profile, so the differences are connectivity's: hops between the ports set the cost of moving the pair, protected or not, and disjoint paths matter only to a check that needs a spare path.

---

## 6. Limitations

**These results describe circuits under one assumed noise model, simulated; they are not a measurement of any device.**

- **One noise model:** Boston's calibration medians. What it leaves out, and why, is in [DESIGN.md, section 4](https://github.com/Rickory2021/bell-transport-bench/blob/submission/docs/DESIGN.md#4-noise-model).
- **One seed per run:** each run takes 100,000 shots per basis, so F's standard error is about 0.0003; a difference under about two combined standard errors needs more shots or seeds before it means anything.
- **Detection, not correction:** the read discards shots and corrects nothing, and no fault injector has tested it fault by fault.
- **Blind to phase flips:** the read checks bit flips only; a phase flip on a port passes it.
- **Idle qubits start in |0⟩:** the transport relies on it, and we have not run this strategy with qubits that may start a shot excited. A qubit that does should read 1 and be discarded, lowering acceptance, but that is untested.
- **The textbook GHZ transport's loss depends on readout:** with no readout error on the qubits it reads it beats ours before any check (appendix A.4), so that comparison holds only for Boston-like readout.
- **No degree penalty:** the XX checks on the contemporary graph in appendix A.2 pass through its five-edge qubit with no extra error; the nine cases avoid that qubit.
- **The graphs:** the contemporary graph is the organisers' 9-node graph, not a real IBM layout, and the hyperbolic-inspired graph is a schematic.

---

## 7. Recommendation

**Empty the path with gates, then read it at the end: a check that costs shots, not gates or qubits.**

On every graph the read raised F above the same transport without it by more than two combined standard errors (section 4.2), and among the circuits we tested that kept at least as many shots as the earlier XX check, it gave the highest F on every graph (appendix A). The same read after the move chain gained less. An XX check on a spare path raised F a little further only by discarding more shots, so for this strategy port distance, not the number of disjoint paths, ordered the protected results (section 4.3). For hardware, readout quality decides between our transport and the textbook GHZ transport that reads the path in X (appendix A.4). The recommendation is conditional on our Boston-based model, and we have not tested this strategy against qubits that start a shot excited.

---

## Appendix A. Each protected case beside its alternatives

**For each graph, one extended benchmark runs the protected case beside the circuits that differ from it by one step, so each comparison shows what one change buys; none fills a case.** Every run shares the nine cases' noise profile, shots and base seed, so the runs the two benchmarks share repeat their numbers exactly. Each section writes out the benchmark file, tables each run against the protected case, shows the circuits section 3.5 does not, each with its pictures folded, then makes the benchmark and sets each run beside its record.

Each run is named for its circuit, followed by the graph; a suffix marks the one run that changes a setting.

| Run | What it is |
|---|---|
| baseline | The organisers' baseline: SWAPs out and back |
| move-chain | The move chain: each hop moves the pair's far half with two CX gates, no check |
| ghz-measure | The textbook GHZ transport: the interior read in X, a Z owed on the first port for each 1, no check |
| ghz-measure, then -readout-0 | The same circuit with no readout error on the interior qubits it reads, the only run that changes the profile |
| ghz-uncompute | GHZ disentangling, the protected case's transport, no check |
| move-chain-vacancy | The move chain, then the vacancy read |
| ghz-uncompute-vacancy | The protected case: GHZ disentangling, then the vacancy read |
| move-chain-path-xx, move-chain-spare-xx | The earlier protected case: the move chain with an XX check, on the path read mid-circuit, or on a spare path |
| ghz-uncompute-vacancy-spare-xx | The protected case, then an XX check on a spare path |

### A.1 The 2016-inspired graph

**The vacancy read raised GHZ disentangling by +0.0023 (5.7 SE), and the same read raised the move chain by +0.0018 (4.3 SE), so the protected case beats the move chain with its read by +0.0010 (2.4 SE).** Without a read the two transports agree within 1.1 combined standard errors. Against the earlier protected case, the protected case is +0.0044 (10.4 SE), keeping 99.3% of shots against 97.0%, with 3 two-qubit gates against 5. The graph has no spare path, so the earlier XX check reads a qubit of the pair's path mid-circuit, under a mid-circuit measurement error of 0.01.

| Run | F ± SE | Acceptance | Two-qubit gates | Against the protected case |
|---|---|---|---|---|
| baseline-ibm2016 | 0.9766 ± 0.0003 | 1.000 | 7 | -0.0075 (-17.1 SE) |
| move-chain-ibm2016 | 0.9814 ± 0.0003 | 1.000 | 3 | -0.0028 (-6.8 SE) |
| ghz-measure-ibm2016 | 0.9790 ± 0.0003 | 1.000 | 2 | -0.0052 (-12.1 SE) |
| ghz-measure-ibm2016-readout-0 | 0.9828 ± 0.0003 | 1.000 | 2 | -0.0013 (-3.3 SE) |
| ghz-uncompute-ibm2016 | 0.9818 ± 0.0003 | 1.000 | 3 | -0.0023 (-5.7 SE) |
| move-chain-vacancy-ibm2016 | 0.9832 ± 0.0003 | 0.994 | 3 | -0.0010 (-2.4 SE) |
| ghz-uncompute-vacancy-ibm2016 | 0.9842 ± 0.0003 | 0.993 | 3 | the protected case |
| move-chain-path-xx-ibm2016 | 0.9797 ± 0.0003 | 0.970 | 5 | -0.0044 (-10.4 SE) |

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-ibm2016.yaml
# config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-ibm2016.yaml

name: deliverable-2026-11-17-extended-ibm2016
label: 2016-inspired, the protected case beside its alternatives
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): on the 2016-inspired graph, the deliverable's protected case,
  case 3, beside the circuits that differ from it by one step, so each comparison
  shows what one change buys. The rungs, each a builder call along the baseline's path
  [0, 3, 4]: the organisers' baseline; the move chain, which moves the pair with two
  CX gates a hop instead of a SWAP's three; ghz_measure, the textbook GHZ transport,
  which reads the interior qubits in X and owes a Z on the first port for each 1;
  ghz_uncompute, which empties the interior with gates instead; ghz_measure again with
  no readout error on the interior qubits it reads, an override only this run takes,
  which shows whether readout error alone decides between the two GHZ transports; each
  of the two empty transports followed by a vacancy read at the end, where only
  ghz_uncompute's last gate makes the read catch a bit flip on the first port; and the
  move-chain benchmark's case 3, the move chain with an XX check on the pair's own
  path, which reads a qubit mid-circuit and runs with the mid-circuit measurement
  error of 0.01 it was reported with. The graph has no spare path, so no rung adds an
  XX check to the vacancy read. Every run takes the deliverable's noise profile, shots
  and base seed, so the runs it shares with the deliverable repeat its numbers
  exactly; the runs it shares with the move-chain benchmark, which takes 20,000 shots,
  agree with them within their errors.
shots: 100000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  # The transports alone, with no check.
  - {run: move-chain-ibm2016, circuit: move-chain-ibm2016, noise: boston-2026}
  - {run: ghz-measure-ibm2016, circuit: ghz-measure-ibm2016, noise: boston-2026}
  # ghz_measure with no readout error on the interior qubits [3], the only run here that changes the profile.
  - {run: ghz-measure-ibm2016-readout-0, circuit: ghz-measure-ibm2016, noise: boston-2026, overrides: [{qubits: 3, field: qubit.readout_error.reads_1_when_0, value: 0.0}, {qubits: 3, field: qubit.readout_error.reads_0_when_1, value: 0.0}]}
  - {run: ghz-uncompute-ibm2016, circuit: ghz-uncompute-ibm2016, noise: boston-2026}
  # A vacancy read at the end after each transport that empties the path; the second is the deliverable's protected case.
  - {run: move-chain-vacancy-ibm2016, circuit: move-chain-vacancy-ibm2016, noise: boston-2026}
  - {run: ghz-uncompute-vacancy-ibm2016, circuit: ghz-uncompute-vacancy-ibm2016, noise: boston-2026}
  # The move-chain benchmark's case 3: the move chain with its XX check on the path, read mid-circuit.
  - {run: move-chain-path-xx-ibm2016, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01}}

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-ibm2016.yaml
# config/circuit/bp.data.circuit.move-chain-ibm2016.yaml

name: move-chain-ibm2016
label: move_chain, ibm2016
processor: ibm2016
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: move_chain().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[4], q[3];

In [ ]:
circuit_views("move-chain-ibm2016", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-measure-ibm2016.yaml
# config/circuit/bp.data.circuit.ghz-measure-ibm2016.yaml

name: ghz-measure-ibm2016
label: ghz_measure, ibm2016
processor: ibm2016
analysis:
  name: parity
  settings:
    checks: []
    frame_z:
    - frame_z_ghz[0]
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: ghz_measure(reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] frame_z_ghz;
  qubit[5] q;
  h q[3];
  cx q[3], q[4];
  cx q[3], q[0];
  h q[3];
  frame_z_ghz[0] = measure q[3];

In [ ]:
circuit_views("ghz-measure-ibm2016", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-vacancy-ibm2016.yaml
# config/circuit/bp.data.circuit.move-chain-vacancy-ibm2016.yaml

name: move-chain-vacancy-ibm2016
label: move_chain then vacancy at the end, ibm2016
processor: ibm2016
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: move_chain();
  then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] check_vac;
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[4], q[3];
  check_vac[0] = measure q[3];

In [ ]:
circuit_views("move-chain-vacancy-ibm2016", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-path-xx-ibm2016.yaml
# config/circuit/bp.data.circuit.move-chain-path-xx-ibm2016.yaml

name: move-chain-path-xx-ibm2016
label: move_chain then bus_xx on the path, ibm2016
processor: ibm2016
analysis:
  name: parity
  settings:
    checks:
    - - check_vac_xx[0]
    - - check_xx[0]
notes: >-
  Built by src/bellbench/builders.py on ibm2016 along the path [0, 3, 4]: move_chain();
  then bus_xx(reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] check_vac_xx;
  bit[1] check_xx;
  qubit[5] q;
  h q[0];
  cx q[0], q[3];
  cx q[3], q[4];
  cx q[4], q[3];
  check_vac_xx[0] = measure q[3];
  h q[3];
  cx q[3], q[0];
  cx q[3], q[4];
  h q[3];
  check_xx[0] = measure q[3];

In [ ]:
circuit_views("move-chain-path-xx-ibm2016", folded=True)

Every file the deliverable-2026-11-17-extended-ibm2016 benchmark uses is now written, so the notebook makes it, draws each run's F, and sets each run beside the run the repository recorded.

In [ ]:
make("deliverable-2026-11-17-extended-ibm2016")
ladder("deliverable-2026-11-17-extended-ibm2016")
beside_record("deliverable-2026-11-17-extended-ibm2016")

### A.2 The contemporary graph

**The vacancy read raised GHZ disentangling by +0.0073 (15.5 SE), and the same read raised the move chain by +0.0045 (9.5 SE), so the protected case beats the move chain with its read by +0.0022 (5.0 SE).** Without a read the two transports agree within 1.0 combined standard errors. Against the earlier protected case, the protected case is +0.0020 (4.5 SE), keeping 97.7% of shots against 97.5%, with 7 two-qubit gates against 11. An XX check on a spare path after the read moves F by +0.0010 (2.2 SE) at an acceptance of 0.956, with 4 more two-qubit gates.

| Run | F ± SE | Acceptance | Two-qubit gates | Against the protected case |
|---|---|---|---|---|
| baseline-contemporary | 0.9600 ± 0.0004 | 1.000 | 19 | -0.0217 (-40.5 SE) |
| move-chain-contemporary | 0.9749 ± 0.0004 | 1.000 | 7 | -0.0068 (-14.6 SE) |
| ghz-measure-contemporary | 0.9663 ± 0.0004 | 1.000 | 4 | -0.0154 (-30.4 SE) |
| ghz-measure-contemporary-readout-0 | 0.9803 ± 0.0003 | 1.000 | 4 | -0.0014 (-3.3 SE) |
| ghz-uncompute-contemporary | 0.9745 ± 0.0004 | 1.000 | 7 | -0.0073 (-15.5 SE) |
| move-chain-vacancy-contemporary | 0.9795 ± 0.0003 | 0.981 | 7 | -0.0022 (-5.0 SE) |
| ghz-uncompute-vacancy-contemporary | 0.9817 ± 0.0003 | 0.977 | 7 | the protected case |
| move-chain-spare-xx-contemporary | 0.9797 ± 0.0003 | 0.975 | 11 | -0.0020 (-4.5 SE) |
| ghz-uncompute-vacancy-spare-xx-contemporary | 0.9827 ± 0.0003 | 0.956 | 11 | +0.0010 (2.2 SE) |

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-contemporary.yaml
# config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-contemporary.yaml

name: deliverable-2026-11-17-extended-contemporary
label: Contemporary, the protected case beside its alternatives
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): on the contemporary graph, the deliverable's protected case,
  case 6, beside the circuits that differ from it by one step, so each comparison
  shows what one change buys. The rungs, each a builder call along the baseline's path
  [0, 1, 2, 3, 7]: the organisers' baseline; the move chain, which moves the pair with
  two CX gates a hop instead of a SWAP's three; ghz_measure, the textbook GHZ
  transport, which reads the interior qubits in X and owes a Z on the first port for
  each 1; ghz_uncompute, which empties the interior with gates instead; ghz_measure
  again with no readout error on the interior qubits it reads, an override only this
  run takes, which shows whether readout error alone decides between the two GHZ
  transports; each of the two empty transports followed by a vacancy read at the end,
  where only ghz_uncompute's last gate makes the read catch a bit flip on the first
  port; and two XX checks on a spare path that shares no qubit with the pair's: the
  move-chain benchmark's case 6, the move chain with the check, and ghz_uncompute with
  its vacancy read followed by the check, which trades discarded shots for F. Both XX
  buses on this graph pass through its five-edge qubit, so their numbers are without
  the planned degree penalty (docs/DESIGN.md, section 4). Every run takes the
  deliverable's noise profile, shots and base seed, so the runs it shares with the
  deliverable repeat its numbers exactly; the runs it shares with the move-chain
  benchmark, which takes 20,000 shots, agree with them within their errors.
shots: 100000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-contemporary, circuit: baseline-contemporary, noise: boston-2026}
  # The transports alone, with no check.
  - {run: move-chain-contemporary, circuit: move-chain-contemporary, noise: boston-2026}
  - {run: ghz-measure-contemporary, circuit: ghz-measure-contemporary, noise: boston-2026}
  # ghz_measure with no readout error on the interior qubits [1, 2, 3], the only run here that changes the profile.
  - {run: ghz-measure-contemporary-readout-0, circuit: ghz-measure-contemporary, noise: boston-2026, overrides: [{qubits: 1, field: qubit.readout_error.reads_1_when_0, value: 0.0}, {qubits: 1, field: qubit.readout_error.reads_0_when_1, value: 0.0}, {qubits: 2, field: qubit.readout_error.reads_1_when_0, value: 0.0}, {qubits: 2, field: qubit.readout_error.reads_0_when_1, value: 0.0}, {qubits: 3, field: qubit.readout_error.reads_1_when_0, value: 0.0}, {qubits: 3, field: qubit.readout_error.reads_0_when_1, value: 0.0}]}
  - {run: ghz-uncompute-contemporary, circuit: ghz-uncompute-contemporary, noise: boston-2026}
  # A vacancy read at the end after each transport that empties the path; the second is the deliverable's protected case.
  - {run: move-chain-vacancy-contemporary, circuit: move-chain-vacancy-contemporary, noise: boston-2026}
  - {run: ghz-uncompute-vacancy-contemporary, circuit: ghz-uncompute-vacancy-contemporary, noise: boston-2026}
  # XX checks on a spare path: the move-chain benchmark's case 6, then the vacancy read followed by the check.
  - {run: move-chain-spare-xx-contemporary, circuit: move-chain-spare-xx-contemporary, noise: boston-2026}
  - {run: ghz-uncompute-vacancy-spare-xx-contemporary, circuit: ghz-uncompute-vacancy-spare-xx-contemporary, noise: boston-2026}

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-contemporary.yaml
# config/circuit/bp.data.circuit.move-chain-contemporary.yaml

name: move-chain-contemporary
label: move_chain, contemporary
processor: contemporary
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  move_chain().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];

In [ ]:
circuit_views("move-chain-contemporary", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-measure-contemporary.yaml
# config/circuit/bp.data.circuit.ghz-measure-contemporary.yaml

name: ghz-measure-contemporary
label: ghz_measure, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks: []
    frame_z:
    - frame_z_ghz[0]
    - frame_z_ghz[1]
    - frame_z_ghz[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  ghz_measure(reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] frame_z_ghz;
  qubit[9] q;
  h q[2];
  cx q[2], q[3];
  cx q[3], q[7];
  cx q[2], q[1];
  cx q[1], q[0];
  h q[1];
  frame_z_ghz[0] = measure q[1];
  h q[2];
  frame_z_ghz[1] = measure q[2];
  h q[3];
  frame_z_ghz[2] = measure q[3];

In [ ]:
circuit_views("ghz-measure-contemporary", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-vacancy-contemporary.yaml
# config/circuit/bp.data.circuit.move-chain-vacancy-contemporary.yaml

name: move-chain-vacancy-contemporary
label: move_chain then vacancy at the end, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
    - - check_vac[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  move_chain(); then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_vac;
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[2];
  check_vac[2] = measure q[3];

In [ ]:
circuit_views("move-chain-vacancy-contemporary", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-spare-xx-contemporary.yaml
# config/circuit/bp.data.circuit.move-chain-spare-xx-contemporary.yaml

name: move-chain-spare-xx-contemporary
label: move_chain then bus_xx on a spare path, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_xx[0]
      - check_xx[1]
      - check_xx[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  move_chain(); then bus_xx(on=[0, 4, 5, 6, 7], reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_xx;
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];
  h q[4];
  cx q[4], q[5];
  cx q[5], q[6];
  cx q[4], q[0];
  cx q[6], q[7];
  h q[4];
  check_xx[0] = measure q[4];
  h q[5];
  check_xx[1] = measure q[5];
  h q[6];
  check_xx[2] = measure q[6];

In [ ]:
circuit_views("move-chain-spare-xx-contemporary", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-spare-xx-contemporary.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-spare-xx-contemporary.yaml

name: ghz-uncompute-vacancy-spare-xx-contemporary
label: ghz_uncompute then vacancy, then bus_xx on a spare path, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
    - - check_vac[2]
    - - check_xx[0]
      - check_xx[1]
      - check_xx[2]
notes: >-
  Built by src/bellbench/builders.py on contemporary along the path [0, 1, 2, 3, 7]:
  ghz_uncompute(); then vacancy(); then bus_xx(on=[0, 4, 5, 6, 7], reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_vac;
  bit[3] check_xx;
  qubit[9] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[2];
  cx q[2], q[3];
  cx q[3], q[7];
  cx q[2], q[3];
  cx q[1], q[2];
  cx q[0], q[1];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[2];
  check_vac[2] = measure q[3];
  h q[4];
  cx q[4], q[5];
  cx q[5], q[6];
  cx q[4], q[0];
  cx q[6], q[7];
  h q[4];
  check_xx[0] = measure q[4];
  h q[5];
  check_xx[1] = measure q[5];
  h q[6];
  check_xx[2] = measure q[6];

In [ ]:
circuit_views("ghz-uncompute-vacancy-spare-xx-contemporary", folded=True)

Every file the deliverable-2026-11-17-extended-contemporary benchmark uses is now written, so the notebook makes it, draws each run's F, and sets each run beside the run the repository recorded.

In [ ]:
make("deliverable-2026-11-17-extended-contemporary")
ladder("deliverable-2026-11-17-extended-contemporary")
beside_record("deliverable-2026-11-17-extended-contemporary")

### A.3 The hyperbolic-inspired graph

**The vacancy read raised GHZ disentangling by +0.0049 (11.2 SE), and the same read raised the move chain by +0.0026 (5.8 SE), so the protected case beats the move chain with its read by +0.0018 (4.3 SE).** Without a read the two transports agree within 1.1 combined standard errors. Against the earlier protected case, the protected case is +0.0022 (5.2 SE), keeping 98.5% of shots against 98.3%, with 5 two-qubit gates against 8. An XX check on a spare path after the read moves F by +0.0013 (3.3 SE) at an acceptance of 0.971, with 3 more two-qubit gates.

| Run | F ± SE | Acceptance | Two-qubit gates | Against the protected case |
|---|---|---|---|---|
| baseline-hyperbolic | 0.9686 ± 0.0004 | 1.000 | 13 | -0.0146 (-29.8 SE) |
| move-chain-hyperbolic | 0.9788 ± 0.0003 | 1.000 | 5 | -0.0044 (-10.1 SE) |
| ghz-measure-hyperbolic | 0.9721 ± 0.0004 | 1.000 | 3 | -0.0111 (-23.5 SE) |
| ghz-measure-hyperbolic-readout-0 | 0.9817 ± 0.0003 | 1.000 | 3 | -0.0014 (-3.4 SE) |
| ghz-uncompute-hyperbolic | 0.9782 ± 0.0003 | 1.000 | 5 | -0.0049 (-11.2 SE) |
| move-chain-vacancy-hyperbolic | 0.9813 ± 0.0003 | 0.988 | 5 | -0.0018 (-4.3 SE) |
| ghz-uncompute-vacancy-hyperbolic | 0.9832 ± 0.0003 | 0.985 | 5 | the protected case |
| move-chain-spare-xx-hyperbolic | 0.9810 ± 0.0003 | 0.983 | 8 | -0.0022 (-5.2 SE) |
| ghz-uncompute-vacancy-spare-xx-hyperbolic | 0.9845 ± 0.0003 | 0.971 | 8 | +0.0013 (3.3 SE) |

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-hyperbolic.yaml
# config/benchmark/bp.data.benchmark.deliverable-2026-11-17-extended-hyperbolic.yaml

name: deliverable-2026-11-17-extended-hyperbolic
label: Hyperbolic-inspired, the protected case beside its alternatives
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): on the hyperbolic-inspired graph, the deliverable's protected
  case, case 9, beside the circuits that differ from it by one step, so each
  comparison shows what one change buys. The rungs, each a builder call along the
  baseline's path [0, 1, 9, 12]: the organisers' baseline; the move chain, which moves
  the pair with two CX gates a hop instead of a SWAP's three; ghz_measure, the
  textbook GHZ transport, which reads the interior qubits in X and owes a Z on the
  first port for each 1; ghz_uncompute, which empties the interior with gates instead;
  ghz_measure again with no readout error on the interior qubits it reads, an override
  only this run takes, which shows whether readout error alone decides between the two
  GHZ transports; each of the two empty transports followed by a vacancy read at the
  end, where only ghz_uncompute's last gate makes the read catch a bit flip on the
  first port; and two XX checks on a spare path that shares no qubit with the pair's:
  the move-chain benchmark's case 9, the move chain with the check, and ghz_uncompute
  with its vacancy read followed by the check, which trades discarded shots for F.
  Every run takes the deliverable's noise profile, shots and base seed, so the runs it
  shares with the deliverable repeat its numbers exactly; the runs it shares with the
  move-chain benchmark, which takes 20,000 shots, agree with them within their errors.
shots: 100000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-hyperbolic, circuit: baseline-hyperbolic, noise: boston-2026}
  # The transports alone, with no check.
  - {run: move-chain-hyperbolic, circuit: move-chain-hyperbolic, noise: boston-2026}
  - {run: ghz-measure-hyperbolic, circuit: ghz-measure-hyperbolic, noise: boston-2026}
  # ghz_measure with no readout error on the interior qubits [1, 9], the only run here that changes the profile.
  - {run: ghz-measure-hyperbolic-readout-0, circuit: ghz-measure-hyperbolic, noise: boston-2026, overrides: [{qubits: 1, field: qubit.readout_error.reads_1_when_0, value: 0.0}, {qubits: 1, field: qubit.readout_error.reads_0_when_1, value: 0.0}, {qubits: 9, field: qubit.readout_error.reads_1_when_0, value: 0.0}, {qubits: 9, field: qubit.readout_error.reads_0_when_1, value: 0.0}]}
  - {run: ghz-uncompute-hyperbolic, circuit: ghz-uncompute-hyperbolic, noise: boston-2026}
  # A vacancy read at the end after each transport that empties the path; the second is the deliverable's protected case.
  - {run: move-chain-vacancy-hyperbolic, circuit: move-chain-vacancy-hyperbolic, noise: boston-2026}
  - {run: ghz-uncompute-vacancy-hyperbolic, circuit: ghz-uncompute-vacancy-hyperbolic, noise: boston-2026}
  # XX checks on a spare path: the move-chain benchmark's case 9, then the vacancy read followed by the check.
  - {run: move-chain-spare-xx-hyperbolic, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026}
  - {run: ghz-uncompute-vacancy-spare-xx-hyperbolic, circuit: ghz-uncompute-vacancy-spare-xx-hyperbolic, noise: boston-2026}

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-chain-hyperbolic.yaml

name: move-chain-hyperbolic
label: move_chain, hyperbolic
processor: hyperbolic
analysis:
  name: postselect
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: move_chain().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[1];
  cx q[9], q[12];
  cx q[12], q[9];

In [ ]:
circuit_views("move-chain-hyperbolic", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-measure-hyperbolic.yaml
# config/circuit/bp.data.circuit.ghz-measure-hyperbolic.yaml

name: ghz-measure-hyperbolic
label: ghz_measure, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks: []
    frame_z:
    - frame_z_ghz[0]
    - frame_z_ghz[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: ghz_measure(reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] frame_z_ghz;
  qubit[13] q;
  h q[1];
  cx q[1], q[9];
  cx q[9], q[12];
  cx q[1], q[0];
  h q[1];
  frame_z_ghz[0] = measure q[1];
  h q[9];
  frame_z_ghz[1] = measure q[9];

In [ ]:
circuit_views("ghz-measure-hyperbolic", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-vacancy-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-chain-vacancy-hyperbolic.yaml

name: move-chain-vacancy-hyperbolic
label: move_chain then vacancy at the end, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: move_chain();
  then vacancy().
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_vac;
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[1];
  cx q[9], q[12];
  cx q[12], q[9];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[9];

In [ ]:
circuit_views("move-chain-vacancy-hyperbolic", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-chain-spare-xx-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-chain-spare-xx-hyperbolic.yaml

name: move-chain-spare-xx-hyperbolic
label: move_chain then bus_xx on a spare path, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_xx[0]
      - check_xx[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: move_chain();
  then bus_xx(on=[0, 3, 11, 12], reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_xx;
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[1];
  cx q[9], q[12];
  cx q[12], q[9];
  h q[3];
  cx q[3], q[11];
  cx q[3], q[0];
  cx q[11], q[12];
  h q[3];
  check_xx[0] = measure q[3];
  h q[11];
  check_xx[1] = measure q[11];

In [ ]:
circuit_views("move-chain-spare-xx-hyperbolic", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.ghz-uncompute-vacancy-spare-xx-hyperbolic.yaml
# config/circuit/bp.data.circuit.ghz-uncompute-vacancy-spare-xx-hyperbolic.yaml

name: ghz-uncompute-vacancy-spare-xx-hyperbolic
label: ghz_uncompute then vacancy, then bus_xx on a spare path, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_vac[0]
    - - check_vac[1]
    - - check_xx[0]
      - check_xx[1]
notes: >-
  Built by src/bellbench/builders.py on hyperbolic along the path [0, 1, 9, 12]: ghz_uncompute();
  then vacancy(); then bus_xx(on=[0, 3, 11, 12], reset=False).
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_vac;
  bit[2] check_xx;
  qubit[13] q;
  h q[0];
  cx q[0], q[1];
  cx q[1], q[9];
  cx q[9], q[12];
  cx q[1], q[9];
  cx q[0], q[1];
  check_vac[0] = measure q[1];
  check_vac[1] = measure q[9];
  h q[3];
  cx q[3], q[11];
  cx q[3], q[0];
  cx q[11], q[12];
  h q[3];
  check_xx[0] = measure q[3];
  h q[11];
  check_xx[1] = measure q[11];

In [ ]:
circuit_views("ghz-uncompute-vacancy-spare-xx-hyperbolic", folded=True)

Every file the deliverable-2026-11-17-extended-hyperbolic benchmark uses is now written, so the notebook makes it, draws each run's F, and sets each run beside the run the repository recorded.

In [ ]:
make("deliverable-2026-11-17-extended-hyperbolic")
ladder("deliverable-2026-11-17-extended-hyperbolic")
beside_record("deliverable-2026-11-17-extended-hyperbolic")

### A.4 Readout and the two GHZ transports

**Under Boston's readout errors the textbook GHZ transport, which reads the interior in X, loses to GHZ disentangling on every graph; with no readout error on the qubits it reads, it wins on every graph.** Each misread interior qubit flips the Z owed on the first port, which leaves the pair in the wrong Bell state, so the loss grows with the qubits it reads: one on the 2016-inspired graph, two on the hyperbolic-inspired and three on the contemporary. Neither run has a check; both keep every shot.

| Graph | Qubits read | ghz-measure minus ghz-uncompute, as modelled | ghz-measure minus ghz-uncompute, no readout error on the read qubits |
|---|---|---|---|
| 2016-inspired | 1 | -0.0028 (-6.4 SE) | +0.0010 (2.4 SE) |
| Contemporary | 3 | -0.0082 (-15.2 SE) | +0.0058 (12.3 SE) |
| Hyperbolic-inspired | 2 | -0.0062 (-12.4 SE) | +0.0035 (7.8 SE) |

---

## Appendix B. Config files nothing above uses

**Every config file nothing above writes: the other noise profiles, the earlier benchmarks and their circuits.** They complete the repository's config, so every assumption the benchmark holds is in this notebook. Each circuit is compiled here as section 3.5 compiles the nine cases' circuits, with its pictures folded; nothing here is run.

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.biased-noise.yaml
# config/benchmark/bp.data.benchmark.biased-noise.yaml

name: biased-noise
label: Biased-noise study
notes: >-
  An optional study, kept apart from the core benchmark (docs/CHALLENGE.md, What is
  fixed and what we choose): if a processor's errors were mostly phase flips and its
  CX kept them so, would an XX check catch nearly every error on the pair? Each graph
  runs the organisers' baseline, move_chain alone, and move_chain with bus_xx, under
  three pauli profiles that differ only in the native two-qubit gate and the noise
  bias: pauli-cx-biased (CX, eta 100), pauli-cz-biased (CZ, eta 100) and
  pauli-cx-unbiased (CX, eta 0.5). On the contemporary and hyperbolic graphs bus_xx
  runs on a spare path and reads only at the end; ibm2016 has no spare path, so there
  it runs on the path, reads each qubit before reusing it, and needs the mid-circuit
  measurement channel, set to 0.01 as one point of the range it is to be swept over.
  The profiles have no readout error, decay or durations, so these numbers describe
  gate noise alone, on one finite model each. 20,000 shots per basis, as in the core.
shots: 20000
base_seed: 20261116
runs:
  # Under pauli-cx-biased.
  - {run: baseline-ibm2016-cx-biased, circuit: baseline-ibm2016, noise: pauli-cx-biased}
  - {run: baseline-contemporary-cx-biased, circuit: baseline-contemporary, noise: pauli-cx-biased}
  - {run: baseline-hyperbolic-cx-biased, circuit: baseline-hyperbolic, noise: pauli-cx-biased}
  - {run: move-ibm2016-cx-biased, circuit: move-chain-ibm2016, noise: pauli-cx-biased}
  - {run: move-contemporary-cx-biased, circuit: move-chain-contemporary, noise: pauli-cx-biased}
  - {run: move-hyperbolic-cx-biased, circuit: move-chain-hyperbolic, noise: pauli-cx-biased}
  - {run: path-xx-ibm2016-cx-biased, circuit: move-chain-path-xx-ibm2016, noise: pauli-cx-biased, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: spare-xx-contemporary-cx-biased, circuit: move-chain-spare-xx-contemporary, noise: pauli-cx-biased}
  - {run: spare-xx-hyperbolic-cx-biased, circuit: move-chain-spare-xx-hyperbolic, noise: pauli-cx-biased}
  # Under pauli-cz-biased.
  - {run: baseline-ibm2016-cz-biased, circuit: baseline-ibm2016, noise: pauli-cz-biased}
  - {run: baseline-contemporary-cz-biased, circuit: baseline-contemporary, noise: pauli-cz-biased}
  - {run: baseline-hyperbolic-cz-biased, circuit: baseline-hyperbolic, noise: pauli-cz-biased}
  - {run: move-ibm2016-cz-biased, circuit: move-chain-ibm2016, noise: pauli-cz-biased}
  - {run: move-contemporary-cz-biased, circuit: move-chain-contemporary, noise: pauli-cz-biased}
  - {run: move-hyperbolic-cz-biased, circuit: move-chain-hyperbolic, noise: pauli-cz-biased}
  - {run: path-xx-ibm2016-cz-biased, circuit: move-chain-path-xx-ibm2016, noise: pauli-cz-biased, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: spare-xx-contemporary-cz-biased, circuit: move-chain-spare-xx-contemporary, noise: pauli-cz-biased}
  - {run: spare-xx-hyperbolic-cz-biased, circuit: move-chain-spare-xx-hyperbolic, noise: pauli-cz-biased}
  # Under pauli-cx-unbiased.
  - {run: baseline-ibm2016-cx-unbiased, circuit: baseline-ibm2016, noise: pauli-cx-unbiased}
  - {run: baseline-contemporary-cx-unbiased, circuit: baseline-contemporary, noise: pauli-cx-unbiased}
  - {run: baseline-hyperbolic-cx-unbiased, circuit: baseline-hyperbolic, noise: pauli-cx-unbiased}
  - {run: move-ibm2016-cx-unbiased, circuit: move-chain-ibm2016, noise: pauli-cx-unbiased}
  - {run: move-contemporary-cx-unbiased, circuit: move-chain-contemporary, noise: pauli-cx-unbiased}
  - {run: move-hyperbolic-cx-unbiased, circuit: move-chain-hyperbolic, noise: pauli-cx-unbiased}
  - {run: path-xx-ibm2016-cx-unbiased, circuit: move-chain-path-xx-ibm2016, noise: pauli-cx-unbiased, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: spare-xx-contemporary-cx-unbiased, circuit: move-chain-spare-xx-contemporary, noise: pauli-cx-unbiased}
  - {run: spare-xx-hyperbolic-cx-unbiased, circuit: move-chain-spare-xx-hyperbolic, noise: pauli-cx-unbiased}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.mid-circuit-sweep.yaml
# config/benchmark/bp.data.benchmark.mid-circuit-sweep.yaml

name: mid-circuit-sweep
label: Mid-circuit measurement error sweep on ibm2016
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): how the XX check on ibm2016 fares as its mid-circuit measurement
  error grows. ibm2016 has no spare path, so its check sits on the pair's own path,
  reads each bus qubit mid-circuit and reuses it, and the move-chain benchmark's case 3
  sets the mid-circuit measurement error to 0.01, one assumed value. Here that circuit,
  move_chain with bus_xx on the pair's path, runs under boston-2026 at mid-circuit
  measurement errors 0, 0.001, 0.003, 0.01, 0.02 and 0.05, each run named by its value
  in e notation after mcm. Each value is one point of a sweep, not a claimed property
  of any device, and each run derives its own seeds, so neighbouring points also differ
  by shot noise: read each difference against its standard errors. The run at 0.01 has
  case 3's settings and seeds, so it gives case 3's numbers. The organisers' baseline
  and move_chain alone read nothing
  mid-circuit, so the error cannot reach them: each runs once, without the channel, as
  a fixed reference. Setting the check's F after postselection, with its acceptance,
  beside move_chain alone's F at each point shows at which errors, if any, the check
  pays for the shots it discards. 20,000 shots per basis and base seed 20261116, as in
  the core.
shots: 20000
base_seed: 20261116
runs:
  # The fixed references, as the move-chain benchmark runs them.
  - {run: baseline-ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  - {run: move-ibm2016, circuit: move-chain-ibm2016, noise: boston-2026}
  # The move chain with its XX check on the pair's path, one run per mid-circuit measurement error.
  - {run: path-xx-ibm2016-mcm-0, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.0}}
  - {run: path-xx-ibm2016-mcm-1e-3, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.001}}
  - {run: path-xx-ibm2016-mcm-3e-3, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.003}}
  - {run: path-xx-ibm2016-mcm-1e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: path-xx-ibm2016-mcm-2e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.02}}
  - {run: path-xx-ibm2016-mcm-5e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.05}}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.middle-out.yaml
# config/benchmark/bp.data.benchmark.middle-out.yaml

name: middle-out
label: Middle-out study
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): do two variants of the strategy the submission reported then,
  the move chain with an XX check, do better than it? The first, move_middle_out, makes the pair in the middle of the path and moves both
  halves outward at once, so it uses the same gates as move_chain in a shorter
  schedule. The second adds a ZZ check before the XX check on the hyperbolic graph,
  whose third qubit-disjoint path between the ports holds a second spare bus; the ZZ
  check catches a bit flip on one port, which the XX check misses, and runs first so
  that the XX check's parity covers the ZZ check's frame bit. The contemporary graph
  has only two qubit-disjoint paths, so it has no room for a second spare bus, and on
  ibm2016 move_middle_out writes the same circuit as move_chain, so neither graph runs
  that variant. Each new run sits beside the move-chain benchmark's run it varies,
  rerun here under the same circuit, profile and base seed, so with the same seeds and
  the same numbers. The ZZ and XX run's XX bus passes through the centre qubit, so its
  numbers are without the planned degree penalty (docs/DESIGN.md, section 4). Every
  run takes 20,000 shots per basis, as in the core.
shots: 20000
base_seed: 20261116
runs:
  # The reference: the move-chain benchmark's runs these vary.
  - {run: move-contemporary, circuit: move-chain-contemporary, noise: boston-2026}
  - {run: move-hyperbolic, circuit: move-chain-hyperbolic, noise: boston-2026}
  - {run: spare-xx-contemporary, circuit: move-chain-spare-xx-contemporary, noise: boston-2026}
  - {run: spare-xx-hyperbolic, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026}
  # Middle-out, without and with the XX check.
  - {run: middle-out-contemporary, circuit: move-middle-out-contemporary, noise: boston-2026}
  - {run: middle-out-hyperbolic, circuit: move-middle-out-hyperbolic, noise: boston-2026}
  - {run: middle-out-xx-contemporary, circuit: move-middle-out-spare-xx-contemporary, noise: boston-2026}
  - {run: middle-out-xx-hyperbolic, circuit: move-middle-out-spare-xx-hyperbolic, noise: boston-2026}
  # Middle-out with a ZZ check, then the XX check, on the hyperbolic graph's two spare paths.
  - {run: middle-out-zz-xx-hyperbolic, circuit: move-middle-out-spare-zz-xx-hyperbolic, noise: boston-2026}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.move-chain.yaml
# config/benchmark/bp.data.benchmark.move-chain.yaml

name: move-chain
label: Move chain with an XX check, the nine cases
notes: >-
  The challenge's nine cases (docs/CHALLENGE.md, The nine core cases) with one
  protection strategy filled in, under boston-2026 as in the core benchmark, so the
  three graphs differ only in connectivity. Cases 1, 2, 4, 5, 7 and 8 are the
  organisers' baseline, the reference, exactly as the core benchmark runs it. Cases
  3, 6 and 9 are the protection: move_chain carries the pair to the ports with two CX
  gates a hop instead of a SWAP's three, then bus_xx reads the ports' XX parity
  through a bus of extra qubits and postselection discards each shot where it reads
  -1. The check catches a phase flip on one port and is blind to a bit flip, to the
  same flip on both ports and to an error on the bus. On the contemporary and
  hyperbolic graphs the bus sits on a spare path that shares no qubit with the pair
  and reads only at the end; ibm2016 has no spare path, so there the bus sits on the
  pair's own path, reads each qubit as vacant before reusing it, and needs the
  mid-circuit measurement channel, set to 0.01 as one point of the range it is to be
  swept over. The move chain also runs without the check, filling no case, so the
  table separates what the cheaper transport buys from what the check buys. Every
  run takes 20,000 shots per basis, as in the core; protected runs are not yet given
  more shots in proportion to their acceptance.
shots: 20000
base_seed: 20261116
runs:
  # The reference: the organisers' baseline.
  - {run: baseline-ibm2016, circuit: baseline-ibm2016, noise: boston-2026}
  - {run: baseline-contemporary, circuit: baseline-contemporary, noise: boston-2026}
  - {run: baseline-hyperbolic, circuit: baseline-hyperbolic, noise: boston-2026}
  # The move chain alone, filling no case.
  - {run: move-ibm2016, circuit: move-chain-ibm2016, noise: boston-2026}
  - {run: move-contemporary, circuit: move-chain-contemporary, noise: boston-2026}
  - {run: move-hyperbolic, circuit: move-chain-hyperbolic, noise: boston-2026}
  # The protection: the move chain with its XX check.
  - {run: path-xx-ibm2016, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: spare-xx-contemporary, circuit: move-chain-spare-xx-contemporary, noise: boston-2026}
  - {run: spare-xx-hyperbolic, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026}
cases:
  - {case: 1, run: baseline-ibm2016, simulation: ideal}
  - {case: 2, run: baseline-ibm2016, simulation: noisy}
  - {case: 3, run: path-xx-ibm2016, simulation: noisy}
  - {case: 4, run: baseline-contemporary, simulation: ideal}
  - {case: 5, run: baseline-contemporary, simulation: noisy}
  - {case: 6, run: spare-xx-contemporary, simulation: noisy}
  - {case: 7, run: baseline-hyperbolic, simulation: ideal}
  - {case: 8, run: baseline-hyperbolic, simulation: noisy}
  - {case: 9, run: spare-xx-hyperbolic, simulation: noisy}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.starting-excitation.yaml
# config/benchmark/bp.data.benchmark.starting-excitation.yaml

name: starting-excitation
label: Starting excitation sweep on the move-chain benchmark's runs
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): how the move-chain benchmark's runs fare if each qubit may start
  a shot in |1> rather than |0>. The move chain's two-CX move is a SWAP without its
  third CX, so it assumes the qubit it moves onto starts in |0>, and the XX check on a
  spare path trusts its spare qubits to start in |0>; the organisers' baseline swaps
  the pair out and back, which returns an excited qubit on its path to where it
  started. Each of the move-chain benchmark's nine runs, the baseline, the move chain
  alone and the move chain with its XX check on each graph, runs under boston-2026 with
  the starting excitation, the chance that each qubit starts a shot in |1>, at 0,
  0.005, 0.01 and 0.02, each run named by its value in e notation after exc; the check
  on ibm2016 keeps case 3's mid-circuit measurement error of 0.01. A run at 0 differs
  from the move-chain benchmark's run of the same circuit only in its seeds. Each value is one
  point of a sweep, not a claimed property of any device, and each run derives its own
  seeds, so neighbouring points also differ by shot noise: read each difference
  against its standard errors. 20,000 shots per basis and base seed 20261116, as in
  the core.
shots: 20000
base_seed: 20261116
runs:
  # The organisers' baseline.
  - {run: baseline-ibm2016-exc-0, circuit: baseline-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: baseline-ibm2016-exc-5e-3, circuit: baseline-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: baseline-ibm2016-exc-1e-2, circuit: baseline-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: baseline-ibm2016-exc-2e-2, circuit: baseline-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: baseline-contemporary-exc-0, circuit: baseline-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: baseline-contemporary-exc-5e-3, circuit: baseline-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: baseline-contemporary-exc-1e-2, circuit: baseline-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: baseline-contemporary-exc-2e-2, circuit: baseline-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: baseline-hyperbolic-exc-0, circuit: baseline-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: baseline-hyperbolic-exc-5e-3, circuit: baseline-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: baseline-hyperbolic-exc-1e-2, circuit: baseline-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: baseline-hyperbolic-exc-2e-2, circuit: baseline-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  # The move chain alone.
  - {run: move-ibm2016-exc-0, circuit: move-chain-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: move-ibm2016-exc-5e-3, circuit: move-chain-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: move-ibm2016-exc-1e-2, circuit: move-chain-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: move-ibm2016-exc-2e-2, circuit: move-chain-ibm2016, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: move-contemporary-exc-0, circuit: move-chain-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: move-contemporary-exc-5e-3, circuit: move-chain-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: move-contemporary-exc-1e-2, circuit: move-chain-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: move-contemporary-exc-2e-2, circuit: move-chain-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: move-hyperbolic-exc-0, circuit: move-chain-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: move-hyperbolic-exc-5e-3, circuit: move-chain-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: move-hyperbolic-exc-1e-2, circuit: move-chain-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: move-hyperbolic-exc-2e-2, circuit: move-chain-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  # The move chain with its XX check, on ibm2016 with case 3's mid-circuit measurement error.
  - {run: path-xx-ibm2016-exc-0, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01, starting_excitation: 0.0}}
  - {run: path-xx-ibm2016-exc-5e-3, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01, starting_excitation: 0.005}}
  - {run: path-xx-ibm2016-exc-1e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01, starting_excitation: 0.01}}
  - {run: path-xx-ibm2016-exc-2e-2, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01, starting_excitation: 0.02}}
  - {run: spare-xx-contemporary-exc-0, circuit: move-chain-spare-xx-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: spare-xx-contemporary-exc-5e-3, circuit: move-chain-spare-xx-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: spare-xx-contemporary-exc-1e-2, circuit: move-chain-spare-xx-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: spare-xx-contemporary-exc-2e-2, circuit: move-chain-spare-xx-contemporary, noise: boston-2026, conditional: {starting_excitation: 0.02}}
  - {run: spare-xx-hyperbolic-exc-0, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.0}}
  - {run: spare-xx-hyperbolic-exc-5e-3, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.005}}
  - {run: spare-xx-hyperbolic-exc-1e-2, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.01}}
  - {run: spare-xx-hyperbolic-exc-2e-2, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026, conditional: {starting_excitation: 0.02}}

In [ ]:
%%repo_file config/benchmark/bp.data.benchmark.vacancy-read.yaml
# config/benchmark/bp.data.benchmark.vacancy-read.yaml

name: vacancy-read
label: Vacancy-read study
notes: >-
  An optional study, kept apart from the nine cases (docs/CHALLENGE.md, What is fixed
  and what we choose): does a GHZ-and-uncompute transport, followed by a vacancy read
  of the emptied path at the end, do better than the strategy the submission reported
  then, the move chain with an XX check? ghz_uncompute makes a GHZ state along the path, then empties the interior with
  gates controlled from the first port, so the vacancy read at the end also catches a
  bit flip on that port before the last gate. On ibm2016 the vacancy read replaces case
  3's XX check on the path, which reads a qubit mid-circuit and reuses it; on the
  contemporary and hyperbolic graphs the XX check on a spare path that the submission
  reported then follows it.
  Each new run sits beside the move-chain benchmark's protected run on its graph,
  rerun here under the same circuit, profile and base seed, so with the same seeds and
  the same numbers. Case 3's run appears twice: at the mid-circuit measurement error
  of 0.01 it is reported with, and at 0, the Boston profile alone, under which every
  new run here needs no channel. The contemporary runs' XX bus passes through the
  five-edge qubit, so their numbers are without the planned degree penalty
  (docs/DESIGN.md, section 4). Every run takes 20,000 shots per basis, as in the core.
shots: 20000
base_seed: 20261116
runs:
  # The reference: the move-chain benchmark's protected runs, case 3's also without the mid-circuit error.
  - {run: path-xx-ibm2016, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.01}}
  - {run: path-xx-ibm2016-mcm-0, circuit: move-chain-path-xx-ibm2016, noise: boston-2026, conditional: {mid_circuit_measurement_error: 0.0}}
  - {run: spare-xx-contemporary, circuit: move-chain-spare-xx-contemporary, noise: boston-2026}
  - {run: spare-xx-hyperbolic, circuit: move-chain-spare-xx-hyperbolic, noise: boston-2026}
  # GHZ and uncompute, then a vacancy read at the end; then the XX check on a spare path where one exists.
  - {run: ghz-vacancy-ibm2016, circuit: ghz-uncompute-vacancy-ibm2016, noise: boston-2026}
  - {run: ghz-vacancy-xx-contemporary, circuit: ghz-uncompute-vacancy-spare-xx-contemporary, noise: boston-2026}
  - {run: ghz-vacancy-xx-hyperbolic, circuit: ghz-uncompute-vacancy-spare-xx-hyperbolic, noise: boston-2026}

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-contemporary.yaml
# config/circuit/bp.data.circuit.move-middle-out-contemporary.yaml

name: move-middle-out-contemporary
label: move_middle_out, contemporary
processor: contemporary
analysis:
  name: postselect
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[9] q;
  h q[1];
  cx q[1], q[2];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];

In [ ]:
circuit_views("move-middle-out-contemporary", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-middle-out-hyperbolic.yaml

name: move-middle-out-hyperbolic
label: move_middle_out, hyperbolic
processor: hyperbolic
analysis:
  name: postselect
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  qubit[13] q;
  h q[1];
  cx q[1], q[9];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[9], q[12];
  cx q[12], q[9];

In [ ]:
circuit_views("move-middle-out-hyperbolic", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-spare-xx-contemporary.yaml
# config/circuit/bp.data.circuit.move-middle-out-spare-xx-contemporary.yaml

name: move-middle-out-spare-xx-contemporary
label: move_middle_out then bus_xx on a spare path, contemporary
processor: contemporary
analysis:
  name: parity
  settings:
    checks:
    - - check_xx[0]
      - check_xx[1]
      - check_xx[2]
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[3] check_xx;
  qubit[9] q;
  h q[1];
  cx q[1], q[2];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[2], q[3];
  cx q[3], q[2];
  cx q[3], q[7];
  cx q[7], q[3];
  h q[4];
  cx q[4], q[5];
  cx q[5], q[6];
  cx q[4], q[0];
  cx q[6], q[7];
  h q[4];
  check_xx[0] = measure q[4];
  h q[5];
  check_xx[1] = measure q[5];
  h q[6];
  check_xx[2] = measure q[6];

In [ ]:
circuit_views("move-middle-out-spare-xx-contemporary", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-spare-xx-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-middle-out-spare-xx-hyperbolic.yaml

name: move-middle-out-spare-xx-hyperbolic
label: move_middle_out then bus_xx on a spare path, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_xx[0]
      - check_xx[1]
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[2] check_xx;
  qubit[13] q;
  h q[1];
  cx q[1], q[9];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[9], q[12];
  cx q[12], q[9];
  h q[3];
  cx q[3], q[11];
  cx q[3], q[0];
  cx q[11], q[12];
  h q[3];
  check_xx[0] = measure q[3];
  h q[11];
  check_xx[1] = measure q[11];

In [ ]:
circuit_views("move-middle-out-spare-xx-hyperbolic", folded=True)

In [ ]:
%%repo_file config/circuit/bp.data.circuit.move-middle-out-spare-zz-xx-hyperbolic.yaml
# config/circuit/bp.data.circuit.move-middle-out-spare-zz-xx-hyperbolic.yaml

name: move-middle-out-spare-zz-xx-hyperbolic
label: move_middle_out then bus_zz and bus_xx on two spare paths, hyperbolic
processor: hyperbolic
analysis:
  name: parity
  settings:
    checks:
    - - check_zz[0]
    - - check_xx[0]
      - check_xx[1]
      - check_xx[2]
      - frame_z_zz[0]
    frame_z:
    - frame_z_zz[0]
notes: Built in the Workbench from a Qiskit circuit.
circuit: |
  OPENQASM 3.0;
  include "stdgates.inc";
  bit[1] check_zz;
  bit[1] frame_z_zz;
  bit[3] check_xx;
  qubit[13] q;
  h q[1];
  cx q[1], q[9];
  cx q[1], q[0];
  cx q[0], q[1];
  cx q[9], q[12];
  cx q[12], q[9];
  cx q[0], q[3];
  cx q[3], q[11];
  cx q[12], q[11];
  check_zz[0] = measure q[11];
  h q[3];
  frame_z_zz[0] = measure q[3];
  h q[2];
  cx q[2], q[6];
  cx q[6], q[10];
  cx q[2], q[0];
  cx q[10], q[12];
  h q[2];
  check_xx[0] = measure q[2];
  h q[6];
  check_xx[1] = measure q[6];
  h q[10];
  check_xx[2] = measure q[10];

In [ ]:
circuit_views("move-middle-out-spare-zz-xx-hyperbolic", folded=True)

In [ ]:
%%repo_file config/noise/bp.data.noise.burlington-2020.yaml
# config/noise/bp.data.noise.burlington-2020.yaml
#
# Medians of IBM's FakeBurlingtonV2 snapshot, a five-qubit T-graph processor.
# Validated by config/bp.schema.noise.yaml; tests/test_config.py pins every value
# to the snapshot and every comment to the schema.

name: burlington-2020
label: FakeBurlingtonV2 medians, calibrated 2020-06-11
kind: snapshot
reference: qiskit-fake-provider
snapshot:
  class: FakeBurlingtonV2
  package: qiskit-ibm-runtime==0.51.0
  calibrated: "2020-06-11"
  properties: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/burlington/props_burlington.json
  configuration: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/burlington/conf_burlington.json
notes: >-
  Medians over the FakeBurlingtonV2 snapshot's 5 qubits and 8 directed CX
  couplers, as qiskit-ibm-runtime 0.51.0 pins them; it is the earliest calibration
  among the snapshots with the T graph. The snapshot's u1, u2 and u3 gates are
  expressed as rz, sx and x: u2 is one pulse and u3 two. The snapshot has no
  measurement duration, so it is borrowed and marked below. It has no reset data,
  so a circuit that resets cannot run on this profile until a value is chosen.
# Time step in seconds: the hardware's clock tick. Null when the snapshot records none.
time_step: null
qubit:
  # T1 in seconds: how long a qubit holds 1 before relaxing to 0; it is still 1 after time t with probability exp(-t/T1).
  t1_relaxation_time: 82.62697735235098e-6
  # T2 in seconds: how long a qubit keeps the phase between 0 and 1; never more than 2 T1.
  t2_dephasing_time: 93.19832447669017e-6
  readout_error:
    # Readout error: the probability of reading 1 when the qubit was 0.
    reads_1_when_0: 0.023
    # Readout error: the probability of reading 0 when the qubit was 1.
    reads_0_when_1: 0.041
# Native gates, by Qiskit name: the qubits each acts on (1 or 2), its error probability per use, and its duration in seconds.
gates:
  cx: {qubits: 2, error_probability: 0.016097735173324107, duration: 389.3333333333333e-9}
  sx: {qubits: 1, error_probability: 0.0005738102719058851, duration: 53.33333333333333e-9, from_snapshot_gate: u2}
  x: {qubits: 1, error_probability: 0.0011472912855835515, duration: 106.66666666666666e-9, from_snapshot_gate: u3}
  rz: {qubits: 1, error_probability: 0.0, duration: 0.0, from_snapshot_gate: u1}
# Measurement: its duration in seconds. Its errors are the readout errors.
measure: {duration: 5.8133333333333335e-6}
borrowed:
  - field: measure.duration
    value: 5.8133333333333335e-6
    source: "FakeOurenseV2 median (calibrated 2021-01-20), qiskit-ibm-runtime==0.51.0"
    url: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/ourense/props_ourense.json
    why: >-
      FakeBurlingtonV2 records no measurement duration. The earliest T-graph snapshots
      that record one, FakeOurenseV2, FakeValenciaV2 and FakeVigoV2 (all calibrated
      2021-01-20), all give this value.

In [ ]:
%%repo_file config/noise/bp.data.noise.fez-2025.yaml
# config/noise/bp.data.noise.fez-2025.yaml
#
# Medians of IBM's FakeFez snapshot (Heron r2). Validated by
# config/bp.schema.noise.yaml; tests/test_config.py pins every value to the
# snapshot and every comment to the schema.

name: fez-2025
label: FakeFez medians, calibrated 2025-02-26
kind: snapshot
reference: qiskit-fake-provider
snapshot:
  class: FakeFez
  package: qiskit-ibm-runtime==0.51.0
  calibrated: "2025-02-26"
  properties: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/fez/props_fez.json
  configuration: https://github.com/Qiskit/qiskit-ibm-runtime/blob/0.51.0/qiskit_ibm_runtime/fake_provider/backends/fez/conf_fez.json
notes: >-
  Medians over the FakeFez snapshot's 156 qubits and 352 directed CZ couplers, as
  qiskit-ibm-runtime 0.51.0 pins them. Every qubit and coupler of a processor
  graph takes these values.
# Time step in seconds: the hardware's clock tick. Null when the snapshot records none.
time_step: 4.0e-9
qubit:
  # T1 in seconds: how long a qubit holds 1 before relaxing to 0; it is still 1 after time t with probability exp(-t/T1).
  t1_relaxation_time: 144.85526609546147e-6
  # T2 in seconds: how long a qubit keeps the phase between 0 and 1; never more than 2 T1.
  t2_dephasing_time: 87.95068681159162e-6
  readout_error:
    # Readout error: the probability of reading 1 when the qubit was 0.
    reads_1_when_0: 0.00390625
    # Readout error: the probability of reading 0 when the qubit was 1.
    reads_0_when_1: 0.011474609375
# Native gates, by Qiskit name: the qubits each acts on (1 or 2), its error probability per use, and its duration in seconds.
gates:
  cz: {qubits: 2, error_probability: 0.0039033828523107883, duration: 84.0e-9}
  sx: {qubits: 1, error_probability: 0.00022868757107977525, duration: 24.0e-9}
  x: {qubits: 1, error_probability: 0.00022868757107977525, duration: 24.0e-9}
  rz: {qubits: 1, error_probability: 0.0, duration: 0.0}
# Measurement: its duration in seconds. Its errors are the readout errors.
measure: {duration: 1.56e-6}
# Reset to 0: its duration in seconds.
reset: {duration: 1.584e-6}
borrowed: []

In [ ]:
%%repo_file config/noise/bp.data.noise.organiser-cx2.yaml
# config/noise/bp.data.noise.organiser-cx2.yaml
#
# The organisers' noise from their example notebook, used as a cross-check that
# reproduces their recorded fidelities, never as the core noise model. Validated
# by config/bp.schema.noise.yaml; tests/test_config.py pins every comment to the
# schema.

name: organiser-cx2
label: The organisers' 2% CX depolarising noise
kind: depolarising
reference: ideal-and-noisy-example
notes: >-
  The organisers' example notebook, section 4 (the noise model cell): a two-qubit
  depolarising channel of strength 0.02 after every CX, every other gate ideal, no
  readout error and no idle decay. Their recipe uses only H and CX; sdg is added,
  ideal like H, so a run can rotate the ports into the YY basis, which the organisers
  read from the density matrix instead of measuring.
depolarising:
  # The gate each depolarising channel follows.
  after_gate: cx
  # Depolarising strength: with this probability, the gate's qubits are left in a fully random state.
  probability: 0.02
  # The gate set; every gate other than after_gate is ideal.
  gates: [h, cx, sdg]

In [ ]:
%%repo_file config/noise/bp.data.noise.pauli-cx-biased.yaml
# config/noise/bp.data.noise.pauli-cx-biased.yaml
#
# A Pauli noise profile for the biased-noise study, kept apart from the core
# benchmark. Validated by config/bp.schema.noise.yaml; tests/test_config.py pins
# every comment to the schema.

name: pauli-cx-biased
label: Pauli noise biased toward phase flips (eta 100), CX native
kind: pauli
reference: puri-2020-bias-preserving-cx
notes: >-
  A what-if for the biased-noise study, never a core model. After every CX, each of its
  two qubits independently takes a Pauli error with probability 0.00942, split by the
  noise bias eta = p_Z / (p_X + p_Y) as p_X = p_Y = p / (2 (eta + 1)). The probability
  makes a two-qubit gate's total error, 1 - (1 - p)^2, about 0.01875, match
  organiser-cx2's (15/16 of its 0.02). With CX native, a phase flip stays a phase flip
  through the gate, the bias-preserving CX that Puri et al. (2020) propose for
  stabilised cat qubits. Only the CX carries an error, so this is the case where the
  bias reaches the pair intact. Every other gate is ideal, with no readout error, no
  decay and no durations, so a run is not scheduled.
pauli:
  # The gate each Pauli error follows, on every qubit it acts on.
  after_gate: cx
  # The probability that each qubit the gate acts on takes a Pauli error (X, Y or Z), independently of its other qubit.
  probability: 0.00942
  # Noise bias eta = p_Z / (p_X + p_Y), above 0; 0.5 is unbiased, as depolarising noise is.
  bias: 100
  # The gate set; every gate other than after_gate is ideal.
  gates: [h, cx, sdg]

In [ ]:
%%repo_file config/noise/bp.data.noise.pauli-cx-unbiased.yaml
# config/noise/bp.data.noise.pauli-cx-unbiased.yaml
#
# A Pauli noise profile for the biased-noise study, kept apart from the core
# benchmark. Validated by config/bp.schema.noise.yaml; tests/test_config.py pins
# every comment to the schema.

name: pauli-cx-unbiased
label: Unbiased Pauli noise (eta 0.5), CX native
kind: pauli
reference: puri-2020-bias-preserving-cx
notes: >-
  A what-if for the biased-noise study, never a core model. After every CX, each of its
  two qubits independently takes a Pauli error with probability 0.00942, split by the
  noise bias eta = p_Z / (p_X + p_Y) as p_X = p_Y = p / (2 (eta + 1)). The probability
  makes a two-qubit gate's total error, 1 - (1 - p)^2, about 0.01875, match
  organiser-cx2's (15/16 of its 0.02). The control: the same channel structure as
  pauli-cx-biased with X, Y and Z equally likely, which on each qubit is a one-qubit
  depolarising channel, so a difference from pauli-cx-biased comes from the noise bias
  alone. Every other gate is ideal, with no readout error, no decay and no durations, so
  a run is not scheduled.
pauli:
  # The gate each Pauli error follows, on every qubit it acts on.
  after_gate: cx
  # The probability that each qubit the gate acts on takes a Pauli error (X, Y or Z), independently of its other qubit.
  probability: 0.00942
  # Noise bias eta = p_Z / (p_X + p_Y), above 0; 0.5 is unbiased, as depolarising noise is.
  bias: 0.5
  # The gate set; every gate other than after_gate is ideal.
  gates: [h, cx, sdg]

In [ ]:
%%repo_file config/noise/bp.data.noise.pauli-cz-biased.yaml
# config/noise/bp.data.noise.pauli-cz-biased.yaml
#
# A Pauli noise profile for the biased-noise study, kept apart from the core
# benchmark. Validated by config/bp.schema.noise.yaml; tests/test_config.py pins
# every comment to the schema.

name: pauli-cz-biased
label: Pauli noise biased toward phase flips (eta 100), CZ native
kind: pauli
reference: puri-2020-bias-preserving-cx
notes: >-
  A what-if for the biased-noise study, never a core model. After every CZ, each of its
  two qubits independently takes a Pauli error with probability 0.00942, split by the
  noise bias eta = p_Z / (p_X + p_Y) as p_X = p_Y = p / (2 (eta + 1)). The probability
  makes a two-qubit gate's total error, 1 - (1 - p)^2, about 0.01875, match
  organiser-cx2's (15/16 of its 0.02). With CZ native, a CX compiles as H CZ H, and the
  second H turns a phase flip on the CX's target into a bit flip, so the bias is partly
  lost. Same error as pauli-cx-biased, so the two differ only in the native two-qubit
  gate. Every other gate is ideal, with no readout error, no decay and no durations, so
  a run is not scheduled.
pauli:
  # The gate each Pauli error follows, on every qubit it acts on.
  after_gate: cz
  # The probability that each qubit the gate acts on takes a Pauli error (X, Y or Z), independently of its other qubit.
  probability: 0.00942
  # Noise bias eta = p_Z / (p_X + p_Y), above 0; 0.5 is unbiased, as depolarising noise is.
  bias: 100
  # The gate set; every gate other than after_gate is ideal.
  gates: [h, cz, sdg]

[Back to the top](#bell-transport-bench:-moving-a-Bell-pair-across-three-processor-graphs)